In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S05_regularizacion"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 5 — Regularización (RIDGE, LASSO, Elastic Net) y regresión no lineal

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


---

## Objetivos de aprendizaje (Sección 1 del cuaderno)

Al terminar la sesión, el estudiante:

- Comprende el **compromiso sesgo-varianza** y cómo la regularización lo gestiona.
- Aplica **RIDGE**, **LASSO** y **Elastic Net**, y elige λ y α por **validación cruzada**.
- Emplea el **LASSO como mecanismo automático de selección de variables**.
- Modela relaciones **no lineales** con regresión polinómica, splines y GAM.
- Justifica el modelo final comparando **interpretabilidad y desempeño**.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —5.1 a 5.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 4)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **5.1** | ¿Por qué el OLS pierde eficacia con un número elevado de variables? | «Teoría guiada» — el compromiso sesgo-varianza |
| **5.2** | ¿Qué significa penalizar un coeficiente? | «Teoría guiada» — penalización, RIDGE, LASSO, Elastic Net y elección de λ |
| **5.3** | ¿Cómo operan la contracción de RIDGE y la selección de LASSO? | «Teoría guiada» — penalización, RIDGE, LASSO, Elastic Net y elección de λ |
| **5.4** | ¿Cómo se determina la intensidad de la penalización? | «Teoría guiada» — penalización, RIDGE, LASSO, Elastic Net y elección de λ |
| **5.5** | ¿Se sostiene con datos reales? La réplica de Tibshirani (1996) | «La réplica de Tibshirani (1996) sobre `prostate`» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **5.6** | ¿Cuándo se puede confiar en un modelo regularizado? | «Supuestos: cómo identificarlos y corregirlos» |
| **5.7** | ¿Qué procede cuando la relación no es lineal? | «No linealidad: polinomios, *splines* y GAM» |
| **5.8** | ¿Qué decisión habilita? Un modelo con costo por variable | «Laboratorio de negocio: Communities and Crime» — laboratorio, paso 6 |
| **5.9** | ¿Qué no se puede afirmar, y qué sigue en S06? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **corre**: enseña cada paso. Convenciones:

- **❓ Qué se quiere averiguar** abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** precede a cada celda de código; **📖 Cómo se lee esta salida** sigue a cada resultado numérico clave. **💡** añade intuición y **⚠️** marca un supuesto o alerta.
- El alumno **reproduce la mecánica de forma manual** y la verifica contra la librería con `assert`: **🖐️ Cálculo manual**, **🧮 Matemática en el cuerpo** (derivación en LaTeX), **✅ Verificación desde la base** (recomputa el resultado y lo cruza con el Excel) y **🧱 Construcción del pipeline desde cero**.
- **📄 En el paper** indica la procedencia exacta de cada resultado replicado (fuente, sección, tabla, página).
- La **«Sección 8» (Supuestos)** ejecuta los diagnósticos; su teoría vive en la guía de supuestos de la sesión.
- **Valor operativo vs. benchmark:** las cifras que se ejecutan aquí son las del **venv** (mandan); las del **paper/ESL** se citan como *benchmark* etiquetado. Pequeñas diferencias (p. ej. `lcavol` 0.676 vs 0.68) provienen de la implementación (sklearn vs glmnet/R) y caen dentro de tolerancia.
- **Convención Excel:** los resultados del modelo se vuelcan a `resultados/S05_resultados.xlsx` y las **figuras de resultados se generan leyendo ese Excel**. Los bloques de descomposición y de supuestos **no escriben** en él.

## Preparación del entorno

La celda siguiente instala las librerías **solo en Google Colab** (versiones fijadas). En ejecución local se omite automáticamente.

**🔎 Qué hace este código.** Declara las **versiones certificadas** del curso (la matriz de versiones certificada del curso) e instala en **Google Colab** una por una, de modo que si un *tag* no resuelve solo degrada **esa** librería (reintento sin fijar versión) en vez de interrumpir la instalación entera. Imprime siempre la tabla **certificada → instalada**, para que cualquier diferencia de versión quede a la vista antes de leer un número. En ejecución local no instala nada: solo reporta.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q pygam

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


**🔎 Qué hace este código.** Importa las librerías, fija la semilla (`RANDOM_STATE = 42`) para reproducibilidad, define la paleta UPC y el ayudante `mostrar()` (guarda cada figura como PNG y la muestra). El backend `Agg` genera figuras sin ventana gráfica.

In [ ]:
# Configuración e imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")            # backend headless: las figuras se guardan como PNG
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  RidgeCV, LassoCV, ElasticNetCV, lasso_path)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta del curso
UPC_ROJO, UPC_TINTA, UPC_GRIS = "#C8102E", "#1F2A44", "#8A8D8F"
PALETA = [UPC_ROJO, UPC_TINTA, "#E4879C", "#5B6472", "#A31621", "#B0B3B5"]
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG y la muestra en el notebook (compatible con backend Agg)."
    fig.tight_layout()
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(ruta)))

print("Librerías cargadas. scikit-learn listo para regularizar.")

**🔎 Qué hace este código.** Localiza la carpeta `data/` en local o la descarga en Colab, y fija las rutas de `resultados/` (Excel) y `figuras/`. Deja lista la variable `XLSX`, que apunta al Excel de contrato.

In [ ]:
# Localización de rutas de la sesión (funciona en local y en Colab)
def localizar_data():
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path("/content")]:
        for d in [base / "data", base / "Sesiones" / "S05_regularizacion" / "data"]:
            if (d / "prostate.csv").exists():
                return d
    return None

DATA = localizar_data()

if DATA is not None:
    SESION = DATA.parent
    print("Datos locales encontrados en:", DATA)
else:
    # Colab: descargar a una carpeta de trabajo
    SESION = Path("/content/S05"); DATA = SESION / "data"
    DATA.mkdir(parents=True, exist_ok=True)
    print("Descargando datasets (Colab)...")
    # prostate: mirror verificado de ESL
    url_p = ("https://raw.githubusercontent.com/empathy87/"
             "The-Elements-of-Statistical-Learning-Python-Notebooks/master/data/Prostate%20Cancer.txt")
    pd.read_csv(url_p).to_csv(DATA / "prostate.csv", index=False)
    # communities: UCI (datos + nombres ARFF)
    base_uci = "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/"
    nombres = [l.split()[1] for l in
               pd.read_csv(base_uci + "communities.names", header=None, sep="\t")[0]
               if str(l).lower().startswith("@attribute")]
    cc = pd.read_csv(base_uci + "communities.data", header=None, names=nombres, na_values="?")
    cc.to_csv(DATA / "communities.csv", index=False)
    print("Descarga completa.")

RESULTADOS = SESION / "resultados"; RESULTADOS.mkdir(exist_ok=True)
FIGURAS = SESION / "figuras"; FIGURAS.mkdir(exist_ok=True)
XLSX = RESULTADOS / "S05_resultados.xlsx"
print("Resultados ->", RESULTADOS)
print("Figuras    ->", FIGURAS)

## 5.1 a 5.4 — Teoría guiada: por qué el OLS pierde eficacia, qué significa penalizar, cómo operan RIDGE y LASSO y cómo se elige la intensidad (Sección 3 del cuaderno)

Cada bloque combina una idea, una **minidemostración** ejecutable y su **lectura de negocio**.

### El compromiso sesgo-varianza: por qué el OLS falla con muchas variables — capítulo 5.1 (subsección 3.1)

El error de predicción fuera de muestra se descompone en **sesgo²** (modelo demasiado rígido), **varianza** (sensibilidad a la muestra de entrenamiento) y **error irreducible** (ruido). Un modelo con un número elevado de variables ajusta casi a la perfección los datos históricos (bajo sesgo) pero es muy inestable ante datos nuevos (alta varianza): **sobreajuste**. La regularización sacrifica algo de sesgo para recortar mucha varianza.

**❓ Qué se quiere averiguar.** ¿Por qué un modelo que describe casi a la perfección el año pasado decepciona el trimestre siguiente?

- **Qué decide:** cuántos indicadores conviene incluir en un modelo. Si más variables fueran siempre mejores, la respuesta sería «todos los que haya» y el reporte interno con cien indicadores tendría razón.
- **Antes de mirar el resultado:** de las 40 columnas, solo **3** llevan señal. Si el error de **prueba** bajara junto con el de entrenamiento, añadir columnas no tendría costo y no haría falta regularizar nada. Si el de entrenamiento cae hasta **0,00** con las 40 columnas mientras el de prueba supera **6**, entonces el ajuste dentro de muestra es una medida no fiable: mide la adecuación a la muestra, no la capacidad predictiva.

**🔎 Qué hace este código.** Genera datos con solo 3 predictores útiles de 40 y ajusta OLS con un número creciente de columnas, midiendo el MSE de **entrenamiento** y de **prueba**. Ilustra el compromiso sesgo-varianza (supuesto 2.3).

In [ ]:
# Mini-demo: al añadir predictores irrelevantes, el ajuste en entrenamiento mejora
# siempre, pero el error de prueba se dispara (varianza).
rng = np.random.default_rng(RANDOM_STATE)
n, p_signal = 60, 3
X_all = rng.normal(size=(n, 40))
beta = np.zeros(40); beta[:p_signal] = [2.0, -1.5, 1.0]
y_sig = X_all @ beta + rng.normal(scale=1.0, size=n)

filas = []
for p in [3, 5, 10, 20, 30, 40]:
    Xtr, Xte, ytr, yte = train_test_split(X_all[:, :p], y_sig, test_size=0.4, random_state=RANDOM_STATE)
    m = LinearRegression().fit(Xtr, ytr)
    filas.append({"n_predictores": p,
                  "MSE_entrenamiento": mean_squared_error(ytr, m.predict(Xtr)),
                  "MSE_prueba": mean_squared_error(yte, m.predict(Xte))})
tabla_bv = pd.DataFrame(filas)
print(tabla_bv.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 4))
ax.plot(tabla_bv.n_predictores, tabla_bv.MSE_entrenamiento, "o-", color=UPC_TINTA, label="Error de entrenamiento")
ax.plot(tabla_bv.n_predictores, tabla_bv.MSE_prueba, "s-", color=UPC_ROJO, label="Error de prueba")
ax.set_xlabel("Número de predictores (solo 3 son útiles)")
ax.set_ylabel("MSE")
ax.set_title("Añadir variables baja el error de entrenamiento\npero dispara el de prueba (sobreajuste)")
ax.legend()
mostrar(fig, FIGURAS / "S05_fig_sesgo_varianza.png")

**📖 Cómo se lee.** El MSE de **entrenamiento** cae siempre al añadir variables (de 0.94 a 0.00 con 40 columnas), pero el de **prueba** aumenta de forma pronunciada (de 1.65 a >6): eso es **sobreajuste**. La honestidad la da el error de prueba, no el de entrenamiento. La regularización es el mecanismo para desplazarse por esta curva. Marco en la guía de supuestos de la sesión («Sección 2.3»).

**Lectura de negocio.** Es la razón por la que un modelo con cien indicadores muestra un desempeño aparentemente excelente en el reporte interno y decepciona el próximo trimestre. El error de entrenamiento **siempre** mejora al agregar variables; la honestidad la da el error de prueba. La regularización es el mecanismo para desplazarse por esta curva.

### RIDGE (penalización L2): contracción continua — capítulo 5.3 (subsección 3.2)

RIDGE minimiza `RSS + λ·Σβj²`. Al crecer λ, **todos** los coeficientes se contraen hacia cero de forma continua, pero **nunca llegan exactamente a cero**: RIDGE no selecciona variables, las mantiene todas con peso reducido. Es especialmente útil cuando hay predictores correlacionados (reparte el peso entre ellos).

### 🧮 Matemática en el cuerpo — RIDGE: función objetivo y solución cerrada

RIDGE resuelve un problema de mínimos cuadrados **penalizado en L2**:

$$\hat{\beta}^{\,\text{ridge}} = \arg\min_{\beta}\; \underbrace{\sum_{i=1}^{n}\Bigl(y_i - \beta_0 - \sum_{j=1}^{p} x_{ij}\beta_j\Bigr)^{2}}_{\text{RSS}} \;+\; \lambda \sum_{j=1}^{p}\beta_j^{2}.$$

Símbolos: $n$ observaciones, $p$ predictores, $x_{ij}$ el predictor $j$ en la observación $i$, $\beta_0$ el intercepto (**no penalizado**) y $\lambda \ge 0$ la fuerza de la penalización. *(Convención de escala: `sklearn.linear_model.Ridge` minimiza esta expresión **en bruto**, sin normalizar por $n$ — a diferencia del LASSO, que sí divide por $2n$; por eso los $\lambda$ de Ridge y LASSO no son comparables entre sí.)*

Como la penalización $\lambda\lVert\beta\rVert_2^2$ es suave y convexa, el problema tiene **solución cerrada** (a diferencia del LASSO). Derivando respecto de $\beta$ e igualando a cero, con los predictores centrados (el intercepto se estima aparte):

$$\hat{\beta}^{\,\text{ridge}} = \bigl(\mathbf{X}^{\top}\mathbf{X} + \lambda \mathbf{I}\bigr)^{-1}\mathbf{X}^{\top}\mathbf{y}.$$

El término $\lambda\mathbf{I}$ suma $\lambda$ a la diagonal de $\mathbf{X}^{\top}\mathbf{X}$: **regulariza** la inversión aunque $\mathbf{X}^{\top}\mathbf{X}$ sea casi singular (colinealidad). Cuando $\lambda \to 0$ se recupera el OLS; cuando $\lambda \to \infty$ todos los $\beta_j \to 0$.

**Ejemplo numérico, aquí mismo (dos predictores casi idénticos).** Sean dos predictores estandarizados con correlación $0{,}99$; entonces $\mathbf{X}^{\top}\mathbf{X} = \begin{bmatrix}1 & 0{,}99\\ 0{,}99 & 1\end{bmatrix}$ y su determinante vale $1 - 0{,}99^{2} = \mathbf{0{,}0199}$: la matriz está al borde de la singularidad y su inversa —la del OLS— se vuelve numéricamente inestable. Con el peaje del Ridge, $\lambda = 0{,}1$:

$$\mathbf{X}^{\top}\mathbf{X} + \lambda\mathbf{I} = \begin{bmatrix}1{,}1 & 0{,}99\\ 0{,}99 & 1{,}1\end{bmatrix},\qquad \det = 1{,}1^{2} - 0{,}99^{2} = 1{,}21 - 0{,}9801 = \mathbf{0{,}2299}$$

—más de **diez veces mayor**: la matriz se despegó del abismo—. En valores propios se lee igual: los de $\mathbf{X}^{\top}\mathbf{X}$ salen del polinomio característico $(1-\mu)^{2} - 0{,}99^{2} = 0 \Rightarrow \mu = 1{,}99$ y $\mu = 0{,}01$; sumar $\lambda\mathbf{I}$ desplaza cada uno en $\lambda$, así que pasan a $2{,}09$ y $\mathbf{0{,}11}$. Comprobación: $2{,}09 \times 0{,}11 = 0{,}2299$, el mismo determinante. Y el intercepto queda fuera: la $\mathbf{I}$ actúa **solo sobre los $p$ predictores**.

La celda de verificación de abajo comprueba esta fórmula cerrada contra `sklearn` con un `assert` (celda 19).

**❓ Qué se quiere averiguar.** ¿Qué se obtiene al obligar a los coeficientes a ser **más pequeños** de lo que los datos por sí solos sugieren, y qué se paga a cambio?

- **Qué decide:** si el modelo entrega recomendaciones parecidas de un trimestre a otro o cambia de magnitud —a veces de signo— cada vez que llega una muestra nueva. Ese es el trato de la regularización: coeficientes **sesgados** a cambio de **menos varianza**, es decir, se renuncia a acertar en promedio para dejar de bailar con cada remuestreo.
- **Antes de mirar el resultado:** si al crecer λ alguna de las 8 líneas **tocara** el cero, RIDGE serviría además para acortar la lista de datos a recolectar. Si todas se acercan sin llegar a tocarlo —**8 de 8 activas incluso con λ = 1000**—, RIDGE estabiliza pero **no** simplifica: la lista de indicadores sigue entera y quien quiera recortarla necesita otra herramienta.

**🔎 Qué hace este código.** Estandariza 8 predictores y ajusta RIDGE para una malla de λ, guardando los coeficientes. La figura muestra cómo se **contraen** al crecer λ sin llegar a cero.

In [ ]:
# Mini-demo RIDGE: los coeficientes se encogen al crecer lambda, sin anularse
Xd = StandardScaler().fit_transform(X_all[:, :8])
lambdas = np.logspace(-2, 3, 40)
coefs_ridge = np.array([Ridge(alpha=l).fit(Xd, y_sig).coef_ for l in lambdas])

fig, ax = plt.subplots(figsize=(6.6, 4))
for j in range(coefs_ridge.shape[1]):
    ax.plot(lambdas, coefs_ridge[:, j], color=PALETA[j % len(PALETA)])
ax.set_xscale("log")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("λ (fuerza de la penalización, escala log)")
ax.set_ylabel("Coeficiente estandarizado")
ax.set_title("RIDGE: los coeficientes se encogen pero no se anulan")
mostrar(fig, FIGURAS / "S05_fig_ridge_encogimiento.png")
print("Coeficientes ≠ 0 con λ grande (α=1000):", int(np.sum(np.abs(Ridge(alpha=1000).fit(Xd, y_sig).coef_) > 1e-8)), "de 8")

**📖 Cómo se lee.** Al crecer λ las 8 líneas se acercan a cero pero **ninguna lo toca**: incluso con λ = 1000 siguen activas las **8 de 8** variables. RIDGE contrae, no selecciona. Es la corrección de referencia para la multicolinealidad (supuesto 1.2).

**🔎 Qué hace este código.** Comprueba la fórmula cerrada de RIDGE del bloque anterior: calcula `β = (XᵀX + λI)⁻¹Xᵀy` con álgebra lineal explícita y verifica con `assert` que coincide con el `Ridge` de `sklearn` sobre los mismos datos estandarizados.

In [ ]:
# 🧮 Verificación: la fórmula cerrada de RIDGE reproduce a sklearn
lam_r = 10.0
yc = y_sig - y_sig.mean()                        # respuesta centrada (el intercepto va aparte)
beta_cerrada = np.linalg.solve(Xd.T @ Xd + lam_r * np.eye(Xd.shape[1]), Xd.T @ yc)
beta_sklearn = Ridge(alpha=lam_r).fit(Xd, y_sig).coef_
print("beta cerrada  (primeros 3):", np.round(beta_cerrada[:3], 4))
print("beta sklearn  (primeros 3):", np.round(beta_sklearn[:3], 4))
print("maxima diferencia:", f"{np.max(np.abs(beta_cerrada - beta_sklearn)):.2e}")
assert np.allclose(beta_cerrada, beta_sklearn, atol=1e-6), "la formula cerrada debe coincidir con sklearn"
print("OK: (XtX + lambda*I)^-1 Xt y reproduce el RIDGE de sklearn.")

**📖 Cómo se lee.** Los coeficientes de la fórmula cerrada coinciden con los de `sklearn` hasta el error de máquina (~1e-16). No hay ningún artificio de librería: RIDGE es álgebra lineal explícita, y el `assert` deja constancia de que la mecánica reconstruida manualmente es la misma que ejecuta la función. Desarrollo en la guía de supuestos de la sesión (Parte 1, «Sección 1.2»).

**Lectura de negocio.** RIDGE estabiliza los coeficientes bajo multicolinealidad (el problema del VIF de la sesión anterior): en vez de coeficientes de magnitud elevada y signo errático, produce estimaciones moderadas y fiables. Es la opción cuando se cree que "casi todas las variables aportan un poco".

### LASSO (penalización L1): selección automática y la importancia del escalado — capítulo 5.3 (subsección 3.3)

LASSO minimiza `RSS + λ·Σ|βj|`. Al crecer λ, no solo contrae los coeficientes: **lleva algunos coeficientes exactamente a cero**, produciendo un modelo *ralo* que usa solo un subconjunto de variables. Es selección de variables integrada en el ajuste.

**❓ Qué se quiere averiguar.** De todos los indicadores disponibles, ¿cuántos hacen falta realmente, y quién toma esa decisión?

- **Qué decide:** cuántas series hay que comprar, depurar y explicar cada mes. Cada variable que sobrevive arrastra un costo de recolección y de mantenimiento; cada una que cae deja de aparecer en el informe y en el presupuesto.
- **Antes de mirar el resultado:** si al subir λ los coeficientes solo se contrajeran, como en RIDGE, la lista seguiría entera y el recorte tendría que hacerse manualmente. Si alguno queda **exactamente en cero** —el conteo pasa de 8 a 6 y luego a 3—, el método entrega la lista ya recortada. Conviene registrar quién eligió el recorte: no los datos por sí solos, sino el λ que se fijó.

**🔎 Qué hace este código.** Ajusta el LASSO sobre los mismos datos para varios λ y cuenta cuántas variables quedan **activas** (coeficiente ≠ 0). Muestra la selección automática: a más λ, menos variables.

In [ ]:
# Mini-demo LASSO: al crecer lambda, cuenta de variables activas cae a cero
for l in [0.01, 0.05, 0.1, 0.3, 0.6]:
    nz = int(np.sum(np.abs(Lasso(alpha=l, max_iter=100000).fit(Xd, y_sig).coef_) > 1e-8))
    print(f"λ = {l:<4}  ->  variables activas: {nz} de 8")

**📖 Cómo se lee.** Con λ pequeño (0.01) sobreviven las 8 variables; al subir λ el conteo cae a 6, luego a 3. El LASSO **anula** coeficientes exactamente (a diferencia de RIDGE): esa es su selección de variables integrada. El mecanismo es el soft-thresholding (bloque siguiente).

### 🧮 Matemática en el cuerpo — LASSO: penalización L1 y soft-thresholding

El LASSO cambia la penalización L2 por **L1** (suma de valores absolutos). Y antes de manipular nada hay que **declarar la normalización**, porque de ella depende el umbral: `sklearn.linear_model.Lasso` —y por tanto este cuaderno— divide la suma de cuadrados por $2n$:

$$\hat{\beta}^{\,\text{lasso}} = \arg\min_{\beta}\; \frac{1}{2n}\sum_{i=1}^{n}\Bigl(y_i - \beta_0 - \sum_{j=1}^{p} x_{ij}\beta_j\Bigr)^{2} \;+\; \lambda \sum_{j=1}^{p}\lvert\beta_j\rvert.$$

⚠️ **Asimetría de convenciones que conviene no ignorar.** `Ridge` de sklearn **no** normaliza (celda 14) y `Lasso` sí divide por $2n$: los $\lambda$ de uno y otro **no están en la misma escala** y no deben compararse entre sí. Con la normalización $\tfrac{1}{2n}$ el umbral del soft-threshold resulta **exactamente** $\lambda$; sin ella arrastraría constantes y el cálculo manual de la celda 27 no cuadraría con la librería.

El valor absoluto **no es derivable en cero**, así que no hay fórmula cerrada; se resuelve por **descenso por coordenadas**. Con los predictores estandarizados, la actualización de cada coordenada es el **operador de umbral suave** (*soft-thresholding*):

$$\hat{\beta}_j = S(\rho_j,\ \lambda) = \operatorname{sign}(\rho_j)\,\bigl(\lvert\rho_j\rvert - \lambda\bigr)_{+},\qquad \rho_j = \tfrac{1}{n}\,\mathbf{x}_j^{\top}\mathbf{r}_{-j},\qquad \mathbf{r}_{-j} = \mathbf{y} - \sum_{k\neq j}\mathbf{x}_k\beta_k,$$

donde $(\,z\,)_{+} = \max(z, 0)$ y $\mathbf{r}_{-j}$ es el **residuo parcial**: lo que queda de $\mathbf{y}$ tras descontar todas las variables salvo la $j$. **Qué es exactamente $\rho_j$** (definición única en toda la sesión, la misma de la Sección 3.3 de la guía del docente): la **señal** de la coordenada $j$ *antes* de penalizar, es decir el coeficiente de una regresión **simple** del residuo parcial sobre $\mathbf{x}_j$ —como los predictores están estandarizados, $\tfrac{1}{n}\mathbf{x}_j^{\top}\mathbf{x}_j = 1$—; con predictores además **ortogonales** coincide con el coeficiente **OLS pleno** de esa coordenada. No es una «correlación parcial»: es una covarianza normalizada y no está acotada en $[-1,1]$.

La clave es el $(\,\cdot\,)_{+}$: si la señal $\lvert\rho_j\rvert$ **no supera** a $\lambda$, el coeficiente se pone **exactamente en 0** — de ahí la selección automática. RIDGE, en cambio, solo multiplica por un factor $<1$ y **nunca** anula.

### 🖐️ Cálculo manual — el operador de soft-thresholding

Se reconstruye la actualización de una coordenada del LASSO sobre un problema de **una** variable estandarizada y se comprueba, con `assert`, que reproduce el coeficiente de `sklearn`.

In [ ]:
# 🖐️ soft-thresholding de una coordenada, verificado contra sklearn Lasso
def soft_threshold(z, gamma):
    "Operador de umbral suave S(z, gamma) = sign(z)*max(|z| - gamma, 0)."
    return np.sign(z) * max(abs(z) - gamma, 0.0)

rng_st = np.random.default_rng(7)
x1 = rng_st.normal(size=200)
x1 = (x1 - x1.mean()) / x1.std()            # estandarizada (var 1, ddof=0): xt x / n = 1
y1 = 1.3 * x1 + rng_st.normal(scale=0.5, size=200)
y1 = y1 - y1.mean()                          # respuesta centrada (sin intercepto)

lam_l = 0.4
rho = (x1 @ y1) / len(y1)                     # senal rho = x^T r / n (x estandarizada)
beta_mano = soft_threshold(rho, lam_l)
beta_sk = Lasso(alpha=lam_l, fit_intercept=False, max_iter=200000).fit(x1.reshape(-1, 1), y1).coef_[0]
print(f"rho = {rho:.4f}   lambda = {lam_l}")
print(f"beta a mano = S(rho, lambda) = {beta_mano:.6f}")
print(f"beta sklearn                = {beta_sk:.6f}")
assert abs(beta_mano - beta_sk) < 1e-6, "el soft-threshold a mano debe igualar a sklearn"
print(f"con lambda = 1.5: beta a mano = {soft_threshold(rho, 1.5):.4f}  (|rho| < lambda => exactamente 0)")
print("OK: el LASSO de sklearn ES el soft-thresholding de la senal rho.")

**📖 Cómo se lee.** Con $\lambda = 0.4$, como la señal $\lvert\rho\rvert \approx 1.31 > \lambda$, el coeficiente se **contrae** (pierde exactamente $\lambda$). Con $\lambda = 1.5 > \lvert\rho\rvert$, cae **exactamente a 0**: la variable sale del modelo. Ese "recorte a cero" es toda la selección del LASSO, ejecutada coordenada a coordenada. Aquí hay **una sola** variable, así que el residuo parcial $\mathbf{r}_{-j}$ es la propia $\mathbf{y}$ centrada y $\rho$ coincide con el coeficiente OLS —el caso ortogonal de la celda 25—. Detalle en la guía de supuestos de la sesión (Parte 2, «Sección 2.4»).

**El escalado es imprescindible.** La penalización actúa sobre la **magnitud** de los coeficientes, y esa magnitud depende de las **unidades** de cada variable. Sin estandarizar, una variable en unidades grandes se penaliza de forma más débil y el modelo dependería arbitrariamente de las escalas.

**❓ Qué se quiere averiguar.** La lista de variables que el LASSO declara importantes, ¿describe el negocio o describe las **unidades** en que se midieron?

- **Qué está en juego:** si una selección automática se puede firmar sin revisar antes la preparación de los datos. Un cambio de moneda o de escala —dólares a miles de dólares— no debería alterar qué indicadores entran en el informe.
- **Antes de mirar el resultado:** si la selección **no cambiara** al multiplicar una variable por 1000, el método sería inmune a las unidades y la estandarización sería un paso accesorio. Si **cambia** —el coeficiente de la variable en unidades grandes cae a ≈ **0,002**, porque penalizar su magnitud casi no tiene costo—, entonces el modelo depende de la hoja de cálculo y no de la señal. Regla dura de la sesión: `StandardScaler` **siempre** antes de RIDGE, LASSO o Elastic Net.

**🔎 Qué hace este código.** Reescala una variable ×1000 y ajusta el LASSO con y sin estandarizar, para mostrar que la selección **cambia** si no se estandariza. Motiva el requisito 2.1.

In [ ]:
# Mini-demo: mismo dato, una variable re-escalada x1000. Sin escalar cambia la selección.
Xraw = X_all[:, :5].copy()
Xraw[:, 0] = Xraw[:, 0] * 1000.0        # variable 0 en "unidades grandes"

sin_escalar = Lasso(alpha=0.1, max_iter=100000).fit(Xraw, y_sig).coef_
con_escalar = Lasso(alpha=0.1, max_iter=100000).fit(StandardScaler().fit_transform(Xraw), y_sig).coef_
comp = pd.DataFrame({"coef_sin_escalar": sin_escalar, "coef_con_escalar": con_escalar},
                    index=[f"x{j}" for j in range(5)])
print(comp.round(4).to_string())
print("\nRegla operativa: SIEMPRE StandardScaler antes de Ridge/Lasso/Elastic Net.")

**📖 Cómo se lee.** Sin escalar, el coeficiente de la variable en «unidades grandes» (x0) queda ≈ 0.002 —se penaliza casi nada— y el modelo depende de las unidades, no de la señal. Con estandarización los coeficientes vuelven a ser comparables. Regla dura: **siempre `StandardScaler` antes de Ridge/Lasso/Elastic Net**.

### 🖐️ Cálculo manual — la estandarización que la penalización exige

La penalización recae sobre la **magnitud** de $\beta_j$, y esa magnitud depende de las **unidades** de $x_j$. Por eso Ridge/LASSO/Elastic Net **no son invariantes a la escala** (el OLS sí). Se reconstruye $z = (x-\mu)/\sigma$ de forma manual y se verifica contra `StandardScaler`; después se muestra que sin estandarizar cambia qué variable sobrevive.

In [ ]:
# 🖐️ estandarizacion a mano z = (x - mu)/sigma, verificada contra StandardScaler
col = X_all[:, 1]
z_mano = (col - col.mean()) / col.std()                          # ddof=0 (poblacional), como StandardScaler
z_sk = StandardScaler().fit_transform(col.reshape(-1, 1)).ravel()
assert np.allclose(z_mano, z_sk), "z a mano debe igualar a StandardScaler"
print("OK: z = (x - mu)/sigma reproduce StandardScaler:",
      f"media={z_mano.mean():.2e}  var={z_mano.var():.3f}")

# Por que IMPORTA: una variable en 'unidades grandes' se penaliza mas debilmente.
Xg = X_all[:, :5].copy(); Xg[:, 0] *= 1000.0                     # x0 en unidades x1000
c_sin = Lasso(alpha=0.1, max_iter=100000).fit(Xg, y_sig).coef_
c_con = Lasso(alpha=0.1, max_iter=100000).fit(StandardScaler().fit_transform(Xg), y_sig).coef_
print(f"coef de x0 SIN escalar = {c_sin[0]:.5f}   CON escalar = {c_con[0]:.4f}")
print("Sin estandarizar, x0 (unidades grandes) recibe un coeficiente diminuto y la seleccion se distorsiona.")

**📖 Cómo se lee.** La versión manual coincide con `StandardScaler` (media ≈ 0, varianza ≈ 1). El segundo bloque muestra el porqué operativo: al poner `x0` en unidades ×1000, **sin** estandarizar su coeficiente se vuelve muy pequeño (casi no recibe penalización) y la selección deja de reflejar la señal. **⚠️** Es el error operativo número uno al regularizar. Método y fuente en la guía de supuestos de la sesión (Parte 2, «Sección 2.1»).

**Lectura de negocio.** LASSO entrega directamente un modelo interpretable con pocas variables ("de 100 indicadores, unos pocos concentran la señal —62 de 100 en el laboratorio de Communities—"): reduce costes de datos, simplifica el despliegue y facilita la explicación ante reguladores. Olvidar el escalado es el error operativo número uno al regularizar.

### Elastic Net (L1 + L2) y el parámetro de mezcla α — capítulo 5.3 (subsección 3.4)

Elastic Net minimiza `RSS + λ·[α·Σ|βj| + (1−α)·Σβj²]`. El mezclador **α ∈ [0,1]** interpola entre RIDGE (α=0) y LASSO (α=1). Hereda la **selección** del LASSO y la **estabilidad** del RIDGE: con grupos de predictores muy correlacionados, tiende a incluir o descartar el grupo en conjunto (*efecto de agrupamiento*), en lugar de elegir uno al azar.

> En scikit-learn, la fuerza de la penalización es `alpha` (el **λ** de la notación del curso) y la mezcla es `l1_ratio` (el **α** de la notación del curso). Atención al doble uso de la palabra "alpha".

### *Paths* de regularización y elección de λ/α por validación cruzada — capítulo 5.4 (subsección 3.5)

El **path** grafica cómo evoluciona cada coeficiente al variar λ. En el LASSO se ve entrar en cero, una a una, las variables: las que **resisten hasta λ grande** son las más importantes. El valor de λ (y de α) **no se fija de forma arbitraria**: se elige por **validación cruzada** (`RidgeCV`, `LassoCV`, `ElasticNetCV`). Dos criterios habituales: **λ_min** (menor error de CV) y **λ_1se** (el mayor λ dentro de un error estándar, se=sd/√k, del mínimo → modelo más simple).

### No linealidad: polinomios, *splines* y GAM — capítulo 5.7 (subsección 3.6)

Cuando la relación no es lineal se puede: añadir potencias (**regresión polinómica**), ajustar polinomios por tramos unidos suavemente (**splines**), o dejar que cada variable tenga una forma libre dentro de un modelo aditivo (**GAM**, *modelo aditivo generalizado*). El GAM conserva la lectura "efecto de cada variable por separado" pero permite que ese efecto sea una curva: es un modelo de "caja de cristal".

**🔎 Qué hace este código.** Sobre una relación no lineal (seno con ruido), ajusta polinomios de grado 1 a 9 y elige el grado por **validación cruzada**, no por el ajuste in-sample. Diagnostica el supuesto 3.1.

In [ ]:
# Mini-demo no lineal: elegir el grado del polinomio por validación cruzada
x_demo = np.linspace(-3, 3, 80)
y_demo = np.sin(x_demo) + rng.normal(scale=0.3, size=x_demo.size)
Xdemo = x_demo.reshape(-1, 1)
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)
grados = range(1, 10)
mse_cv = [(-cross_val_score(make_pipeline(PolynomialFeatures(g), LinearRegression()),
                            Xdemo, y_demo, cv=cv, scoring="neg_mean_squared_error")).mean()
          for g in grados]
g_opt = list(grados)[int(np.argmin(mse_cv))]
print("Grado óptimo por CV:", g_opt)

fig, ax = plt.subplots(figsize=(6.2, 4))
ax.plot(list(grados), mse_cv, "o-", color=UPC_ROJO)
ax.axvline(g_opt, color=UPC_TINTA, ls="--", label=f"grado óptimo = {g_opt}")
ax.set_xlabel("Grado del polinomio")
ax.set_ylabel("MSE de validación cruzada")
ax.set_title("El grado se elige por CV, no por el ajuste in-sample")
ax.legend()
mostrar(fig, FIGURAS / "S05_fig_grado_polinomio_cv.png")

**📖 Cómo se lee.** El grado que minimiza el error de CV es **6**. Elegirlo por el ajuste in-sample llevaría a un grado excesivo que **oscila en los extremos** (fenómeno de Runge) y predice absurdos fuera del rango. Nunca se elige el grado por el R² de entrenamiento. Método en la guía de supuestos de la sesión («Sección 3.1»).

**🔎 Qué hace este código.** Registra el error de validación cruzada de los polinomios de **grado 1 a 7** para la relación **real** `lpsa ~ lcavol` de `prostate` —no la del seno sintético de la celda anterior— con **siete semillas de barajado** distintas, y calcula el **error estándar** (se = sd/√k) del mejor grado. Es el cálculo que sostiene el **Drill 3**, y sirve para decidir si «el grado 1 es el mejor» es una conclusión o un accidente de la partición. Se vuelca a la hoja `drill3_grado_semillas` del Excel.

In [ ]:
# REGISTRO Drill 3: grado del polinomio por CV en `lpsa ~ lcavol` (relacion REAL), semilla a semilla
prost_d3 = pd.read_csv(DATA / "prostate.csv")
x_d3 = prost_d3[["lcavol"]].to_numpy()
y_d3 = prost_d3["lpsa"].to_numpy()
GRADOS_D3 = list(range(1, 8))
SEMILLAS_D3 = [42, 0, 1, 7, 13, 123, 2024]

filas_d3, se_d3_42 = [], {}
for semilla in SEMILLAS_D3:
    kf = KFold(n_splits=10, shuffle=True, random_state=semilla)
    medias = []
    for grado in GRADOS_D3:
        pipe = make_pipeline(PolynomialFeatures(grado), StandardScaler(), LinearRegression())
        err = -cross_val_score(pipe, x_d3, y_d3, cv=kf, scoring="neg_mean_squared_error")
        medias.append(float(err.mean()))
        if semilla == 42:
            se_d3_42[grado] = float(err.std() / np.sqrt(10))
    filas_d3.append([semilla] + [round(m, 4) for m in medias] + [GRADOS_D3[int(np.argmin(medias))]])

tabla_drill3 = pd.DataFrame(filas_d3,
                            columns=["semilla"] + [f"grado_{g}" for g in GRADOS_D3] + ["argmin"])
grado_argmin_42 = int(tabla_drill3.loc[0, "argmin"])
cv_argmin_42 = float(tabla_drill3.loc[0, f"grado_{grado_argmin_42}"])
se_argmin_42 = se_d3_42[grado_argmin_42]
techo_1se_d3 = cv_argmin_42 + se_argmin_42
grados_dentro_1se = [g for g in GRADOS_D3 if float(tabla_drill3.loc[0, f"grado_{g}"]) <= techo_1se_d3]

print(tabla_drill3.to_string(index=False))
print(f"\nSemilla 42: el minimo esta en el grado {grado_argmin_42} (CV-MSE {cv_argmin_42:.4f}), se = {se_argmin_42:.4f}")
print(f"Techo de un error estandar = {techo_1se_d3:.4f}  ->  grados INDISTINGUIBLES del mejor: {grados_dentro_1se}")
print(f"Grado ganador segun la semilla: {dict(zip(tabla_drill3['semilla'], tabla_drill3['argmin']))}")

**📖 Cómo se lee.** Con la semilla 42 el mínimo está en el **grado 1** (CV-MSE 0.6379), pero el error estándar es **se ≈ 0.058**: los grados **1, 2, 3, 4, 5 y 7** caen **dentro de un error estándar** del mejor y son estadísticamente **indistinguibles**. Solo el **grado 6** se separa con claridad (0.7652, a ~2,2 se). Al cambiar la semilla del barajado el ganador se mueve: con las semillas **0 y 123** el mínimo es el **grado 3**. **⚠️** La lectura defendible no es «la CV elige el grado 1», sino «la CV descarta el grado 6 y no distingue entre los grados bajos; se elige el **más simple** dentro de la banda, que es el grado 1». Registro completo en la hoja `drill3_grado_semillas`.

**Lectura de negocio.** Muchas relaciones económicas no son lineales (rendimientos decrecientes, saturación, umbrales). La CV protege contra la tentación de subir el grado buscando un R² in-sample mayor: casi siempre empeora la predicción fuera de muestra. El polinomio global tiene, además, un defecto conocido: oscila de forma errática en los extremos del rango (fenómeno de Runge). Los **splines** y los **GAM** resuelven esa rigidez.

#### Splines: polinomios por tramos

Un **spline** ajusta polinomios de grado bajo (típicamente cúbicos) en tramos separados por *nudos*, exigiendo que se unan de forma suave. Así se logra una curva flexible sin recurrir a un único polinomio global de grado alto, que se vuelve inestable en los extremos. A continuación se ajusta una regresión con **B-splines** (base `bs` de `patsy`, resuelta con `statsmodels`) para la relación no lineal `lpsa ~ lcavol` del propio `prostate`.

**🔎 Qué hace este código.** Ajusta una regresión con **B-splines cúbicos** (base `bs` de `patsy`, resuelta con `statsmodels`) para la relación no lineal `lpsa ~ lcavol` de `prostate`, reutilizando los mismos nudos al predecir sobre una malla fina.

In [ ]:
# Demo SPLINES: regresión con B-splines cúbicos sobre una relación no lineal de `prostate`
import statsmodels.api as sm
from patsy import dmatrix, build_design_matrices

df_nl = pd.read_csv(DATA / "prostate.csv").sort_values("lcavol")
x_nl = df_nl["lcavol"].to_numpy()
y_nl = df_nl["lpsa"].to_numpy()

# Base de B-splines cúbicos (df=6 -> nudos internos por cuantiles); patsy añade el intercepto
base_bs = dmatrix("bs(x, df=6, degree=3)", {"x": x_nl}, return_type="dataframe")
modelo_spline = sm.OLS(y_nl, base_bs).fit()

# Predicción sobre una malla fina reutilizando los MISMOS nudos (build_design_matrices)
grid = np.linspace(x_nl.min(), x_nl.max(), 200)
base_grid = build_design_matrices([base_bs.design_info], {"x": grid})[0]
base_grid = pd.DataFrame(np.asarray(base_grid), columns=base_bs.design_info.column_names)
y_hat_spline = modelo_spline.predict(base_grid)

print("B-splines (grado 3, df=6) para lpsa ~ lcavol")
print(f"  R² del ajuste  = {modelo_spline.rsquared:.3f}")
print(f"  funciones base = {base_bs.shape[1] - 1} (+ intercepto)")

fig, ax = plt.subplots(figsize=(6.6, 4))
ax.scatter(x_nl, y_nl, color=UPC_GRIS, s=25, alpha=0.7, label="Observaciones")
ax.plot(grid, y_hat_spline, color=UPC_ROJO, lw=2.2, label="Ajuste con B-splines (grado 3)")
ax.set_xlabel("lcavol (log del volumen del tumor)")
ax.set_ylabel("lpsa (log del PSA)")
ax.set_title("Splines: polinomios cúbicos por tramos unidos suavemente")
ax.legend()
mostrar(fig, FIGURAS / "S05_fig_splines_lcavol.png")

**📖 Cómo se lee.** El spline captura la curvatura de `lpsa ~ lcavol` con un R² de ajuste de **0.557** usando 6 funciones base, sin las oscilaciones de un polinomio global de grado alto. Ajusta tramos locales unidos con suavidad: estable en las colas.

**🔎 Qué hace este código.** Ajusta la misma relación `lpsa ~ lcavol` con una base de **splines naturales** (`cr` de `patsy`, *natural cubic regression spline*), que es la que el sílabo prescribe junto a los B-splines. La diferencia con `bs` no está en el ajuste sino en las **colas**: la base natural **impone que la curva sea recta más allá de los nudos extremos**, de modo que se puede evaluar fuera del rango observado sin las oscilaciones que hacen inestable a un polinomio global. Se registra en la hoja `splines_naturales` del Excel.

In [ ]:
# Demo SPLINES NATURALES (base `cr` de patsy): cubicos por tramos y RECTOS fuera de los nudos extremos
base_ns = dmatrix("cr(x, df=6)", {"x": x_nl}, return_type="dataframe")
modelo_ns = sm.OLS(y_nl, base_ns).fit()

base_ns4 = dmatrix("cr(x, df=4)", {"x": x_nl}, return_type="dataframe")
modelo_ns4 = sm.OLS(y_nl, base_ns4).fit()

# Malla que SE SALE del rango observado a proposito: solo la base natural admite evaluarla
grid_ext = np.linspace(x_nl.min() - 0.8, x_nl.max() + 0.8, 240)
base_ext = build_design_matrices([base_ns.design_info], {"x": grid_ext})[0]
base_ext = pd.DataFrame(np.asarray(base_ext), columns=base_ns.design_info.column_names)
y_hat_ns = np.asarray(modelo_ns.predict(base_ext))

tabla_splines = pd.DataFrame(
    [["bs (B-spline cubico)", 6, int(modelo_spline.df_model), round(float(modelo_spline.rsquared), 4),
      int(base_bs.shape[1]), "no"],
     ["cr (spline natural)", 6, int(modelo_ns.df_model), round(float(modelo_ns.rsquared), 4),
      int(base_ns.shape[1]), "si"],
     ["cr (spline natural)", 4, int(modelo_ns4.df_model), round(float(modelo_ns4.rsquared), 4),
      int(base_ns4.shape[1]), "si"]],
    columns=["base", "df_pedido", "gl_modelo", "R2_ajuste", "n_columnas_base", "admite_extrapolar"])
print(tabla_splines.to_string(index=False))
print(f"\nSpline NATURAL (df=6): R2 = {modelo_ns.rsquared:.4f} con {int(modelo_ns.df_model)} grados de libertad.")
print(f"B-spline (df=6):       R2 = {modelo_spline.rsquared:.4f} con {int(modelo_spline.df_model)} grados de libertad.")
print("Mismo ajuste con un parametro menos, y ademas evaluable fuera del rango observado.")

fig, ax = plt.subplots(figsize=(6.8, 4.1))
ax.scatter(x_nl, y_nl, color=UPC_GRIS, s=25, alpha=0.7, label="Observaciones")
ax.plot(grid, y_hat_spline, color=UPC_GRIS, lw=1.8, ls="--", label="B-spline `bs` (df=6)")
ax.plot(grid_ext, y_hat_ns, color=UPC_ROJO, lw=2.2, label="Spline natural `cr` (df=6)")
ax.axvline(x_nl.min(), color=UPC_TINTA, lw=0.9, ls=":")
ax.axvline(x_nl.max(), color=UPC_TINTA, lw=0.9, ls=":", label="Rango observado de lcavol")
ax.set_xlabel("lcavol (log del volumen del tumor)")
ax.set_ylabel("lpsa (log del PSA)")
ax.set_title("Spline natural: recto fuera de los nudos extremos")
ax.legend(fontsize=8)
mostrar(fig, FIGURAS / "S05_fig_splines_naturales.png")

**📖 Cómo se lee.** El spline **natural** alcanza un R² de **0.5566** con **5** grados de libertad, frente al **0.5567** con **6** del B-spline: el mismo ajuste con un parámetro menos. Fuera de las líneas punteadas —el rango observado de `lcavol`— la curva roja se prolonga en **línea recta**: es la restricción de linealidad en las colas que define a la base natural y la razón por la que se prefiere cuando hay que **extrapolar** o cuando los extremos tienen pocos datos. **💡** Es la corrección que los supuestos de la sesión prescriben para la no linealidad con colas ralas (la guía de supuestos de la sesión, «Sección 3.1»).

#### GAM: modelo aditivo generalizado

Un **GAM** modela la respuesta como una **suma de funciones suaves**, una por predictor: `lpsa = β₀ + f₁(lcavol) + f₂(lweight) + f₃(pgg45)`. Cada `f` es un spline penalizado que aprende su propia forma, de modo que se capturan no linealidades **sin perder** la lectura "efecto de cada variable por separado". A continuación se ajusta un `LinearGAM` de `pyGAM` con términos suaves `s(...)` sobre tres predictores continuos de `prostate`.

**🔎 Qué hace este código.** Ajusta un `LinearGAM` de `pyGAM` con una función suave `s(...)` por predictor (`lcavol`, `lweight`, `pgg45`) sobre `prostate` y grafica la **dependencia parcial** de cada uno con su banda del 95 %. Cada término `s(...)` lleva un parámetro de suavizado `lam` (por defecto **0.6** en `pyGAM`) que penaliza la **curvatura** de la spline —la integral de su segunda derivada—, igual que λ penaliza la magnitud de los coeficientes en RIDGE/LASSO: un `lam` grande endereza la curva (más sesgo, menos varianza) y uno pequeño la deja ondular. Aquí se usa el valor por defecto, del que resulta el pseudo-R² **0.734**.

In [ ]:
# Demo GAM: LinearGAM de pyGAM con términos suaves s(...) sobre 3 predictores de `prostate`
from pygam import LinearGAM, s

df_gam = pd.read_csv(DATA / "prostate.csv")
pred_gam = ["lcavol", "lweight", "pgg45"]
Xg = df_gam[pred_gam].to_numpy()
yg = df_gam["lpsa"].to_numpy()

gam = LinearGAM(s(0) + s(1) + s(2)).fit(Xg, yg)   # una función suave por predictor
gam.summary()   # resumen: grados de libertad efectivos (EDoF), significancia y pseudo-R²
pseudo_r2_gam = gam.statistics_["pseudo_r2"]["explained_deviance"]
print(f"\nPseudo-R² del GAM = {pseudo_r2_gam:.3f}")

# Figura de dependencias parciales: la forma suave estimada para cada predictor
fig, axes = plt.subplots(1, len(pred_gam), figsize=(12, 4))
etiquetas = {"lcavol": "lcavol (log volumen tumor)",
             "lweight": "lweight (log peso próstata)",
             "pgg45": "pgg45 (% Gleason 4/5)"}
for i, ax in enumerate(axes):
    XX = gam.generate_X_grid(term=i)
    pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)
    ax.plot(XX[:, i], pdep, color=UPC_ROJO, lw=2)
    ax.fill_between(XX[:, i], confi[:, 0], confi[:, 1], color=UPC_ROJO, alpha=0.15)
    ax.set_title(f"s({pred_gam[i]})")
    ax.set_xlabel(etiquetas[pred_gam[i]])
axes[0].set_ylabel("Efecto parcial sobre lpsa")
fig.suptitle("GAM: forma suave estimada para cada predictor (± banda del 95 %)", y=1.03)
mostrar(fig, FIGURAS / "S05_fig_gam_dependencias.png")

**📖 Cómo se lee.** El GAM alcanza un pseudo-R² de **0.734** conservando la lectura «efecto de cada variable por separado»: `s(lcavol)` es fuerte y significativo, `s(lweight)` moderado y `s(pgg45)` **plano y no significativo** (p ≈ 0.13). Es el modelo de «caja de cristal»: flexible e interpretable a la vez. (El aviso de `pyGAM` sobre p-valores es esperable con suavizado estimado.)

**Lectura de negocio.** Frente al polinomio **global** —que impone una única forma rígida a todo el rango y oscila de forma inestable en los extremos—, los **splines** ajustan tramos locales y producen curvas suaves y estables. El **GAM** lleva esa idea a varios predictores a la vez conservando la **interpretabilidad**: cada dependencia parcial se lee por separado ("a mayor `lcavol`, mayor `lpsa`, con rendimientos casi lineales"; el efecto de `pgg45` es plano y no significativo) sin sacrificar la flexibilidad. Es el modelo de "caja de cristal" recomendable cuando se necesita capturar no linealidades y, a la vez, explicar el efecto de cada variable ante un comité o un regulador.

---

## 5.5 — ¿Se sostiene con datos reales? La réplica de Tibshirani (1996) sobre `prostate`

**Contexto.** Tibshirani (1996, *Regression Shrinkage and Selection via the Lasso*, JRSS-B 58(1)) presentó el LASSO usando el dataset de próstata de Stamey et al. (1989): 97 varones, se predice el log del antígeno prostático específico (`lpsa`) a partir de 8 mediciones clínicas. *The Elements of Statistical Learning* (ESL, 2ª ed., Sección 3.4, **Tabla 3.3**) publica la comparación canónica OLS / Ridge / Lasso que aquí se reproduce.

**Resultados a reproducir (ESL Tabla 3.3):** coeficiente OLS de `lcavol` ≈ **0.68**; MSE de prueba del OLS ≈ **0.521**; el LASSO conserva **3 variables** {`lcavol`, `lweight`, `svi`} con MSE de prueba ≈ **0.479**.

### 📄 En el paper — de dónde proviene esta réplica (subsección 4.0)

- **Dataset `prostate`:** Stamey et al. (1989), *J. Urology* 141(5):1076–1083 (97 varones; se predice `lpsa`).
- **Tabla de coeficientes que se reproduce:** Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning* (ESL), 2ª ed. (2009), **«Sección 3.4» (Shrinkage Methods), Tabla 3.3, p. 63**; la partición train/test (67/30) se describe en la **p. 48**.
- **Paper seminal del LASSO:** Tibshirani, R. (1996), *Regression Shrinkage and Selection via the Lasso*, **JRSS-B 58(1):267–288**, DOI 10.1111/j.2517-6161.1996.tb02080.x.
- **Base original: sí** (mirror fiel `empathy87/ESL`, `data/Prostate Cancer.txt`; el enlace de Hastie da HTTP-403 a scripts). Detalle en la ficha de la sesión de réplica del paper.

### Qué preguntaba Tibshirani, y por qué usó lo que usó — Sección 0 del paper (subsección 4.0)

**💡 Antes de tocar los datos.** Una réplica sin esta pregunta se convierte en mecánica: se ejecutan celdas y se obtiene un número. Lo que sigue explica **qué buscaba el autor** y **por qué eligió cada pieza de su método**, que es de donde proviene el criterio para elegir un método propio en el futuro. *(Desarrollo completo con las citas del original: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El LASSO no nació del escenario de «una gran cantidad de variables».** El ejemplo con el que Tibshirani demostró el método tiene **ocho predictores y noventa y siete pacientes**: un problema donde el OLS es perfectamente estimable y donde, aun así, su respuesta no le sirve a nadie. La pregunta del artículo no es «¿cómo estimar cuando no se puede?», sino **«¿por qué el analista queda insatisfecho con la respuesta del OLS incluso cuando el OLS funciona?»**. Los datos vienen de un estudio clínico sobre el **diagnóstico y el tratamiento** del adenocarcinoma de próstata (Stamey et al., 1989), y ahí un modelo con ocho coeficientes distintos de cero no dice qué mediciones vale la pena tomar.

**El objetivo declarado, y sus dos mitades** (resumen, p. 267). «The "lasso" minimizes the residual sum of squares subject to the sum of the absolute value of the coefficients being less than a constant. Because of the nature of this constraint it tends to produce some coefficients that are exactly 0 and hence gives interpretable models.» La Sección 1 del artículo desglosa la insatisfacción con el OLS en **dos motivos**: la **precisión de predicción** —poco sesgo, mucha varianza; contraer o anular coeficientes puede mejorar el error fuera de muestra— y la **interpretación** —el analista quiere el subconjunto más pequeño con los efectos más fuertes—. El artículo persigue **las dos a la vez**, y esa conjunción es la contribución.

**Por qué ni la selección de subconjuntos ni el Ridge, y por qué tampoco el garrote.** La **selección de subconjuntos** entrega un modelo legible, pero es un **proceso discreto** —cada regresor entra o sale— y por eso cambios pequeños en los datos producen modelos muy distintos: varianza alta, y una «interpretación» que no interpreta nada si cambia al re-muestrear. El **Ridge** (Hoerl y Kennard, 1970) es un **proceso continuo** que contrae los coeficientes y estabiliza, pero **no lleva ningún coeficiente a 0**. Cada uno atiende un motivo y compromete el otro. Existía además el *nonnegative garrote* de **Breiman (1995)**, que ya contraía y anulaba a la vez —«subset selection is unstable, ridge is very stable, and the nn-garrote is intermediate»—; el LASSO tenía que ser mejor que él, y lo es porque el garrote **reescala los coeficientes del OLS** en lugar de estimarlos desde cero, de modo que hereda su mal comportamiento justo donde se quiere regularizar. El LASSO evita el uso explícito de los estimadores del OLS.

**Por qué L1, y por qué estandarizar antes.** Ambos métodos minimizan la suma de cuadrados **sujeta a un presupuesto** sobre el tamaño de los coeficientes, y la forma del presupuesto lo decide todo: el Ridge restringe la suma de **cuadrados** (frontera lisa, un círculo, sin esquinas) y el LASSO la suma de **valores absolutos** (frontera en **rombo**, con vértices sobre los ejes, y un vértice es un punto donde una coordenada vale **cero exacto**). El exponente 1 es el caso límite: por debajo la región deja de ser convexa, por encima pierde los vértices. Y como la penalización recae sobre la **magnitud** de los coeficientes, que depende de las **unidades**, los ocho predictores se estandarizan antes de ajustar —sin escala común el método seleccionaría por sistema de medida, no por importancia—. Es la celda de estandarización que viene a continuación, y no un trámite.

**Por qué ESE conjunto de datos.** El artículo **no documenta la razón de su elección** y aquí no se le inventa una. Sí constan cuatro propiedades verificables que lo vuelven una prueba justa: procedencia clínica de Stanford; p = 8 y n = 97, de modo que el OLS es estimable y el LASSO no compite con un método defectuoso; con ocho predictores la búsqueda exhaustiva recorre 2⁸ = 256 modelos y era computable en 1996, así que el rival estándar podía presentarse en su mejor versión; y una estructura de correlación sin colinealidad severa, lo que deja claro que el argumento es la **parsimonia**.

**⚠️ Qué se reproduce, y qué no.** Lo que se replica es la **conclusión** —un modelo ralo cuyos supervivientes son `{lcavol, lweight, svi}` con errores de prueba indistinguibles entre sí frente al error estándar publicado—, no el decimal exacto de una implementación de 1996. Que el conteo publicado oscile entre 3 y 4 variables, o que un coeficiente difiera en la tercera cifra, proviene del **criterio de λ** y de la aritmética de cada librería: son diferencias de procedimiento, no errores. Y conviene decirlo con la misma honestidad que el artículo: el LASSO **no destaca por precisión** —su error es equivalente dentro del ruido—; su ventaja es la parsimonia.


**🔎 Qué hace este código.** Carga `prostate.csv` y verifica sus dimensiones y la partición fija de ESL (columna `train`).

In [ ]:
# Paso 1: cargar y verificar el dataset
prost = pd.read_csv(DATA / "prostate.csv")
print("Dimensiones:", prost.shape)               # (97, 11)
print("Partición train/test:", prost["train"].value_counts().to_dict())   # T=67, F=30
prost.head(3)

**📖 Cómo se lee.** El dataset tiene **97 filas × 11 columnas** y la partición **T = 67 / F = 30** exacta de ESL (p. 48). No se rebaraja: se usa la columna `train` tal cual para poder comparar con la tabla del libro.

**🔎 Qué hace este código.** Estandariza los 8 predictores con la **convención de ESL** (escalador ajustado sobre las 97 filas), separa `y = lpsa` y las máscaras train/test, y comprueba que el train escalado queda en media≈0 y varianza≈1.

In [ ]:
# Paso 2: estandarización — CONVENCIÓN DE ESL (escalar los 8 predictores sobre las 97 filas)
predictores = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
tr = (prost["train"] == "T").values
te = (prost["train"] == "F").values

escala_esl = StandardScaler().fit(prost[predictores].values)   # sobre las 97 observaciones
X = escala_esl.transform(prost[predictores].values)
y = prost["lpsa"].values
Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
print(f"Entrenamiento: {Xtr.shape[0]} filas, Prueba: {Xte.shape[0]} filas")
print("Media de X_train escalado ≈ 0 y varianza ≈ 1 (comprobación):",
      np.round(Xtr.mean(), 3), np.round(Xtr.var(), 3))

**📖 Cómo se lee.** El train escalado queda en media ≈ 0 y varianza ≈ 1: la estandarización es correcta. **⚠️ Nota de convención:** ESL ajusta el escalador sobre las **97 filas** para reproducir la Tabla 3.3; un flujo de ML «puro» lo ajustaría **solo con el train** (como en el laboratorio de Communities). El coeficiente de `lcavol` cambia (0.676 vs 0.711) pero el MSE de prueba del OLS **no** (el OLS es invariante a la escala). En cambio, RIDGE/LASSO/Elastic Net **sí** dependen de la escala, así que ajustar el escalador sobre las 97 filas introduce una **fuga leve** del test en la estandarización; se acepta para reproducir exactamente la Tabla 3.3 (el laboratorio de Communities usa el flujo sin fuga: escalador **solo en train**). Ver la guía de supuestos de la sesión («Sección 2.1») y «Sección 8» de este cuaderno.

**🔎 Qué hace este código.** Ajusta el OLS sobre los predictores estandarizados, lee el coeficiente de `lcavol`, confirma que es el mayor en valor absoluto y calcula el MSE de prueba y su error estándar.

In [ ]:
# Paso 3: OLS — coeficiente de lcavol y error de prueba
ols = LinearRegression().fit(Xtr, ytr)
coef_lcavol_ols = ols.coef_[predictores.index("lcavol")]
mse_test_ols = mean_squared_error(yte, ols.predict(Xte))

coef_ols = pd.Series(ols.coef_, index=predictores)
print("Coeficientes OLS (predictores estandarizados):")
print(coef_ols.round(3).to_string())
print(f"\n>> lcavol = {coef_lcavol_ols:.3f}  (ESL Tabla 3.3: 0.68)")
print(f">> Es el mayor en valor absoluto: {coef_ols.abs().idxmax() == 'lcavol'}")
print(f">> MSE de prueba del OLS = {mse_test_ols:.3f}  (ESL: 0.521)")
# Error estándar del error de prueba
err2 = (yte - ols.predict(Xte)) ** 2
print(f">> Std Error del MSE de prueba = {err2.std(ddof=1)/np.sqrt(len(yte)):.3f}  (ESL: 0.179)")

**📖 Cómo se lee.** El coeficiente de `lcavol` es **0.676** (venv; *benchmark* ESL: 0.68) y es el **mayor en valor absoluto** → el predictor dominante. El MSE de prueba del OLS es **0.521** (ESL: 0.521) con error estándar **0.179** (ESL: 0.179). La diferencia 0.676 vs 0.68 es el efecto del `ddof`: el escalador se ajusta sobre las **97 filas** (`ddof=0` de sklearn frente a `ddof=1` de R), un factor √(97/96) ≈ **0.52 %**, dentro de tolerancia.

### 📄 En el paper — coeficiente OLS de `lcavol`

ESL Tabla 3.3 (p. 63), columna **LS**: `lcavol` = **0.68**, *Test Error* = **0.521** (Std Error 0.179). El valor operativo del venv (0.676 / 0.521) reproduce ambos dentro de la tolerancia declarada en la ficha de la sesión de réplica del paper («Sección 4», targets #1 y #3).

**🔎 Qué hace este código.** Ajusta RIDGE eligiendo λ por validación cruzada (`RidgeCV`) y reporta el λ óptimo, el coeficiente de `lcavol`, el nº de variables activas y el MSE de prueba.

In [ ]:
# Paso 4: RIDGE por validación cruzada (RidgeCV) — no anula ninguna variable
ridge = RidgeCV(alphas=np.logspace(-3, 3, 200)).fit(Xtr, ytr)
mse_test_ridge = mean_squared_error(yte, ridge.predict(Xte))
n_nz_ridge = int(np.sum(np.abs(ridge.coef_) > 1e-8))
print(f"RIDGE: λ óptimo = {ridge.alpha_:.3f}")
print(f"  coef lcavol = {ridge.coef_[0]:.3f}  (ESL: 0.588; encogido respecto al OLS)")
print(f"  variables activas = {n_nz_ridge} de 8  (RIDGE NO selecciona)")
print(f"  MSE de prueba = {mse_test_ridge:.3f}  (ESL: 0.496; mejora al OLS)")

**📖 Cómo se lee.** RIDGE elige λ ≈ **4.15**, contrae `lcavol` a **0.596** (ESL: 0.588) pero mantiene las **8 de 8** variables (RIDGE no selecciona), y baja el MSE de prueba a **0.497** (ESL: 0.496): mejora al OLS pleno. Contracción continua, sin ceros.

**❓ Qué se quiere averiguar.** ¿Se entrega el modelo **más preciso** o el modelo **más simple**? Los datos ofrecen los dos y no eligen entre ellos.

- **Qué decide:** cuántas variables lleva el modelo que se despliega, con su costo de recolección y su facilidad de defensa ante un comité. `λ_min` optimiza el error de CV; `λ_1se` acepta un error algo mayor a cambio de menos variables.
- **Antes de mirar el resultado:** si ambos criterios cayeran sobre el mismo λ, no habría dilema. Si `λ_1se` resulta claramente mayor y deja **menos** variables a cambio de un CV-MSE que aún queda **dentro de un error estándar** del mínimo, los dos modelos son estadísticamente indistinguibles y la elección deja de ser técnica: pasa a ser una decisión de negocio entre desempeño y parsimonia.

**🔎 Qué hace este código.** Ajusta el LASSO eligiendo λ por CV (`LassoCV`) y reporta dos criterios: **λ_min** (menor error de CV) y **λ_1se** (mayor λ dentro de un error estándar, se=sd/√k, del mínimo → modelo más simple). Dos argumentos técnicos gobiernan la convergencia y, con ella, **qué coeficiente se declara cero** (el núcleo de la selección): `max_iter=200000` es el tope de iteraciones del descenso por coordenadas —se fija alto para que el solver **converja del todo** sobre datos estandarizados y no quede un `ConvergenceWarning` con el conjunto activo sin estabilizar—; `tol=1e-4` (valor por defecto) es la tolerancia de parada (se detiene cuando la mejora del objetivo cae por debajo de ese umbral). Un coeficiente cuenta como **activo** solo si `|βⱼ| > 1e-8`; `tol` y ese umbral, juntos, fijan la frontera entre «cero» y «distinto de cero».

In [ ]:
# Paso 5: LASSO - eleccion de lambda por validacion cruzada con pliegues BARAJADOS
# El archivo `prostate.csv` viene ORDENADO por la respuesta (`lpsa` creciente). Con `cv=10`
# (un entero) scikit-learn construye `KFold` SIN barajar y cada pliegue queda siendo una
# franja contigua de `lpsa`: la validacion cruzada deja de ser representativa. Se baraja.
cv_barajada = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

lasso_cv = LassoCV(cv=cv_barajada, random_state=RANDOM_STATE, max_iter=200000, n_alphas=200).fit(Xtr, ytr)

mse_media = lasso_cv.mse_path_.mean(axis=1)                                    # error de CV por lambda
mse_se = lasso_cv.mse_path_.std(axis=1) / np.sqrt(lasso_cv.mse_path_.shape[1])  # se = sd/raiz(k)
i_min = int(np.argmin(mse_media))
umbral = mse_media[i_min] + mse_se[i_min]
lambda_1se = lasso_cv.alphas_[mse_media <= umbral].max()
i_1se = int(np.where(lasso_cv.alphas_ == lambda_1se)[0][0])
lasso_1se = Lasso(alpha=lambda_1se, max_iter=200000).fit(Xtr, ytr)
nz_min = np.array(predictores)[np.abs(lasso_cv.coef_) > 1e-8]
nz_1se = np.array(predictores)[np.abs(lasso_1se.coef_) > 1e-8]

# Dos magnitudes DISTINTAS por cada lambda: el CV-MSE (con el que se ELIGE lambda) y el MSE
# de prueba (con el que se JUZGA el modelo ya elegido). Se imprimen rotuladas y por separado.
print(f"lambda_min = {lasso_cv.alpha_:.4f}  ->  {len(nz_min)} variables activas")
print(f"    CV-MSE en lambda_min (error de validacion cruzada) = {mse_media[i_min]:.4f}   (se = {mse_se[i_min]:.4f})")
print(f"    MSE de PRUEBA en lambda_min (n_test = 30)          = {mean_squared_error(yte, lasso_cv.predict(Xte)):.4f}")
print(f"lambda_1se = {lambda_1se:.4f}  ->  {len(nz_1se)} variables activas: {list(map(str, nz_1se))}")
print(f"    CV-MSE en lambda_1se (error de validacion cruzada) = {mse_media[i_1se]:.4f}")
print(f"    MSE de PRUEBA en lambda_1se (n_test = 30)          = {mean_squared_error(yte, lasso_1se.predict(Xte)):.4f}")
print("\nEl lambda que minimiza el error de CV apenas penaliza (conserva casi todas las variables);")
print("la regla de un error estandar entrega un modelo mas simple.")

**📖 Cómo se lee.** Con los pliegues barajados, `λ_min ≈ 0.0129` apenas penaliza (deja **7** variables, CV-MSE **0.5359**) y `λ_1se ≈ 0.1939` es más simple (**5** variables, CV-MSE **0.6483**). **⚠️ Dos magnitudes que nunca se intercambian:** el **CV-MSE** es el error de **validación cruzada**, el que se usa para **elegir** λ; el **MSE de prueba** (0.4942 y 0.4714 respectivamente) es el error sobre los 30 casos reservados, el que se usa para **juzgar** el modelo ya elegido. El **0.4851** de la tabla comparativa (y del Excel) es un valor distinto: el MSE de prueba del **LASSO operativo de la réplica** (λ = 0.2203, 3 variables), elegido para reproducir el modelo ralo de ESL por **parsimonia**, no por minimizar ningún error. Que estas cifras no coincidan no es contradicción: en un test de solo 30 casos, con error estándar ≈ 0.179, **todas estas diferencias caen dentro del ruido**. **💡** Elegir entre λ_min y λ_1se es una **decisión de negocio**: desempeño puro vs. parsimonia/interpretabilidad. Ver la guía de supuestos de la sesión («Sección 2.2» y «Sección 2.5»).

**🔎 Qué hace este código.** Mide **cuánto cambia la elección de λ por barajar los pliegues**, que es la corrección aplicada en la celda anterior. Repite la misma CV de dos formas —`cv=10` (entero: `KFold` **sin barajar**, lo que el material hacía antes) y `KFold(shuffle=True)`— y añade la **correlación entre el orden de las filas y `lpsa`**, que es la causa del problema. Repite además λ_1se con **seis semillas de barajado** para mostrar cuánto se mueve. Se vuelca a la hoja `cv_barajado` del Excel.

In [ ]:
# REGISTRO: que cambia por BARAJAR los pliegues de la CV (hoja `cv_barajado`)
corr_orden_lpsa = float(np.corrcoef(np.arange(len(prost)), prost["lpsa"].values)[0, 1])

def registro_lasso_cv(cv):
    """Devuelve lambda_min / lambda_1se con su CV-MSE, su MSE de prueba y su n de variables."""
    m = LassoCV(cv=cv, random_state=RANDOM_STATE, max_iter=200000, n_alphas=200).fit(Xtr, ytr)
    media = m.mse_path_.mean(axis=1)
    se = m.mse_path_.std(axis=1) / np.sqrt(m.mse_path_.shape[1])
    k = int(np.argmin(media))
    lam_1se = m.alphas_[media <= media[k] + se[k]].max()
    k1 = int(np.where(m.alphas_ == lam_1se)[0][0])
    m1 = Lasso(alpha=lam_1se, max_iter=200000).fit(Xtr, ytr)
    activas = list(np.array(predictores)[np.abs(m1.coef_) > 1e-8])
    return {
        "lambda_min": float(m.alpha_),
        "n_vars_lambda_min": int(np.sum(np.abs(m.coef_) > 1e-8)),
        "cv_mse_lambda_min": float(media[k]),
        "se_cv_lambda_min": float(se[k]),
        "mse_test_lambda_min": float(mean_squared_error(yte, m.predict(Xte))),
        "lambda_1se": float(lam_1se),
        "n_vars_lambda_1se": len(activas),
        "cv_mse_lambda_1se": float(media[k1]),
        "mse_test_lambda_1se": float(mean_squared_error(yte, m1.predict(Xte))),
        "vars_lambda_1se": " ".join(activas),
    }

def registro_enet_cv(cv):
    m = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1.0], cv=cv,
                     random_state=RANDOM_STATE, max_iter=200000).fit(Xtr, ytr)
    return {"enet_l1_ratio": float(m.l1_ratio_), "enet_lambda": float(m.alpha_),
            "enet_n_vars": int(np.sum(np.abs(m.coef_) > 1e-8)),
            "enet_mse_test": float(mean_squared_error(yte, m.predict(Xte)))}

registro_cv_sin = {**registro_lasso_cv(10), **registro_enet_cv(10)}            # KFold SIN barajar
registro_cv_con = {**registro_lasso_cv(cv_barajada), **registro_enet_cv(cv_barajada)}  # barajado, semilla 42

filas_cv_semillas = []
for semilla in [42, 0, 1, 7, 13, 2024]:
    r = registro_lasso_cv(KFold(n_splits=10, shuffle=True, random_state=semilla))
    filas_cv_semillas.append([semilla, round(r["lambda_min"], 4), r["n_vars_lambda_min"],
                              round(r["cv_mse_lambda_min"], 4), round(r["lambda_1se"], 4),
                              r["n_vars_lambda_1se"], round(r["cv_mse_lambda_1se"], 4)])
tabla_cv_semillas = pd.DataFrame(filas_cv_semillas,
                                 columns=["semilla", "lambda_min", "n_vars_min", "cv_mse_min",
                                          "lambda_1se", "n_vars_1se", "cv_mse_1se"])

comparacion_cv = pd.DataFrame({"cv_sin_barajar": registro_cv_sin, "cv_barajado_semilla42": registro_cv_con})
print(f"Correlacion entre el orden de las filas de prostate.csv y `lpsa` = {corr_orden_lpsa:.4f}")
print("(el archivo esta ordenado por la respuesta: sin barajar, cada pliegue es una franja de lpsa)\n")
print(comparacion_cv.to_string())
print("\nlambda_1se con distintas semillas de barajado:")
print(tabla_cv_semillas.to_string(index=False))

**📖 Cómo se lee.** La correlación entre el orden de las filas y `lpsa` es **0.9581**: el archivo está ordenado por la respuesta. Sin barajar, cada pliegue es una franja contigua de `lpsa` y el error de CV se **infla**: CV-MSE mínimo **0.7565** frente a **0.5359** barajando; y λ_min sale en 0.0038 (deja las **8** variables) frente a 0.0129 (deja **7**). En λ_1se, sin barajar quedan **5** variables (λ 0.1418) y barajando con la semilla 42 también **5** (λ 0.1939) —pero con las semillas 0, 1, 7 y 2024 quedan las **3** de la réplica `{lcavol, lweight, svi}`—. El Elastic Net cambia incluso de mezcla: `α` pasa de **0.1** (sin barajar) a **0.5** (barajado). **⚠️ La lección:** cuando el archivo llega ordenado por la respuesta —o por cualquier variable con señal—, `KFold` sin `shuffle=True` produce pliegues no representativos y **la λ elegida es un artefacto del orden del archivo**, no una propiedad de los datos. Desarrollo en la guía de supuestos de la sesión («Sección 2.5»).

**❓ Qué se quiere averiguar.** De las ocho medidas del expediente clínico, ¿cuántas bastan para anticipar el marcador `lpsa`, y cuánta precisión cuesta quedarse solo con esas?

- **Qué decide:** el modelo que se publica como resultado de la réplica. Es la cifra central de la sesión: **3** variables `{lcavol, lweight, svi}` frente a las 8 del OLS, con la pregunta de si el recorte se paga en error.
- **Antes de mirar el resultado:** el OLS con las ocho deja un MSE de prueba de **0,5213**. Si el LASSO de 3 variables saliera claramente peor, la simplicidad se pagaría con precisión y habría que justificarla ante quien use el modelo. Si resulta **igual o mejor** —el *benchmark* etiquetado de ESL es **0,479**—, entonces cinco de las ocho medidas no aportaban nada que las otras tres no dijeran ya.

**🔎 Qué hace este código.** Reproduce el modelo LASSO de 3 variables de ESL: recorre el *path* y toma el λ del **borde ralo** cuyo conjunto activo es exactamente `{lcavol, lweight, svi}`. Este es el **λ operativo del LASSO** de la réplica.

In [ ]:
# Paso 5b: reproducir el modelo LASSO de ESL Tabla 3.3 (3 variables: lcavol, lweight, svi)
# ESL/glmnet elige por CV sobre el factor de encogimiento un modelo más ralo que el λ_1se de
# sklearn. Se reproduce ese modelo tomando el LASSO más ajustado cuyo conjunto activo sea
# exactamente {lcavol, lweight, svi} (borde ralo del path, coherente con ESL).
lambda_esl = None
for a in np.logspace(np.log10(0.05), np.log10(0.5), 400):
    activo = set(np.array(predictores)[np.abs(Lasso(alpha=a, max_iter=200000).fit(Xtr, ytr).coef_) > 1e-8])
    if activo == {"lcavol", "lweight", "svi"}:
        lambda_esl = a
        break

lasso_esl = Lasso(alpha=lambda_esl, max_iter=200000).fit(Xtr, ytr)
nz_esl = np.array(predictores)[np.abs(lasso_esl.coef_) > 1e-8]
lasso_n_nonzero = int(len(nz_esl))
mse_test_lasso = mean_squared_error(yte, lasso_esl.predict(Xte))

print(f"LASSO (réplica ESL): λ = {lambda_esl:.4f}")
print(f"  variables conservadas ({lasso_n_nonzero}): {list(map(str, nz_esl))}")
print("  coeficientes:", {p: round(float(c), 3) for p, c in zip(predictores, lasso_esl.coef_)})
print(f"  MSE de prueba = {mse_test_lasso:.3f}  (ESL Tabla 3.3: 0.479)")

**📖 Cómo se lee.** El λ operativo es **0.2203**; el LASSO conserva **3 variables** `{lcavol, lweight, svi}` (coefs 0.534 / 0.180 / 0.080) con MSE de prueba **0.485** (venv; *benchmark* ESL: 0.479). **⚠️** El λ exacto de ESL proviene de `glmnet` (R); `sklearn` (descenso por coordenadas) sitúa su λ_1se algo menos ralo. Se reproduce la **conclusión** —modelo de 3 variables encabezado por `lcavol`—, no el decimal de glmnet.

**🔎 Qué hace este código.** Ajusta Elastic Net eligiendo por CV el mezclador α (`l1_ratio`) y la fuerza λ (`alpha`), y reporta variables activas y MSE de prueba.

In [ ]:
# Paso 6: Elastic Net por CV (malla de l1_ratio = alpha, y alpha = lambda) - pliegues BARAJADOS
# Misma correccion que en el Paso 5: `cv_barajada` en lugar de `cv=10` (KFold sin barajar sobre
# un archivo ordenado por `lpsa`). Con pliegues representativos la CV elige otra mezcla.
enet = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1.0], cv=cv_barajada,
                    random_state=RANDOM_STATE, max_iter=200000).fit(Xtr, ytr)
mse_test_enet = mean_squared_error(yte, enet.predict(Xte))
n_nz_enet = int(np.sum(np.abs(enet.coef_) > 1e-8))
print(f"Elastic Net: alpha (l1_ratio) = {enet.l1_ratio_} - lambda = {enet.alpha_:.4f}")
print(f"  variables activas = {n_nz_enet} - MSE de prueba = {mse_test_enet:.3f}")
print("  Un alpha cercano a 1 pide seleccion fuerte (como LASSO); cercano a 0, mas encogimiento (como RIDGE).")

**📖 Cómo se lee.** Con los pliegues barajados la CV elige **α = 0.5** (mezcla equilibrada entre L1 y L2) y λ ≈ **0.0213**, dejando **7 variables** con MSE de prueba **0.495**. **⚠️** Sin barajar —el registro previo de la hoja `cv_barajado`— la CV elegía α = 0.1 y λ = 0.0246 (MSE 0.504): la mezcla del Elastic Net es, también ella, sensible a cómo se arman los pliegues. Con predictores correlacionados el término L2 estabiliza la elección (efecto de agrupamiento).

**❓ Qué se quiere averiguar.** Si tres sensores miden casi la misma magnitud, ¿cuál de los tres «importa»? Y sobre todo: un cero del LASSO, ¿significa «esta variable no aporta» o «esta variable ya está dicha por otra»?

- **Por qué es el error caro de la sesión:** de esa lectura depende que se retire un sensor, se dé de baja una fuente de datos o se deje de comprar un indicador porque su coeficiente salió cero.
- **Antes de mirar el resultado:** las tres variables correlacionan **≈ 0,999**. Si el LASSO repartiera el peso entre las tres, el cero se leería sin riesgo. Si conserva **una** y anula las otras dos, y además **cuál** sobrevive cambia con la semilla del ruido, entonces el cero resultó de una elección arbitraria entre variables gemelas: retirar el sensor equivocado deja al modelo sin la señal del grupo entero. El Elastic Net entra en la comparación precisamente por eso.

**🔎 Qué hace este código.** Construye el escenario del **Drill 2** —dos réplicas ruidosas de `lcavol`, «tres sensores que miden casi la misma magnitud»— y lo resuelve con LASSO puro y con Elastic Net para cinco semillas del ruido. Deja explícita la construcción (ruido normal de desviación **0.05** sobre el `lcavol` ya estandarizado, generador `np.random.default_rng`), que es lo que hace **reproducibles** las tablas del drill. Se vuelca a la hoja `drill2_gemelas` del Excel.

In [ ]:
# REGISTRO Drill 2: construccion de las gemelas ruidosas y reparto del peso (LASSO vs Elastic Net)
SD_RUIDO_D2 = 0.05          # ruido pequeño: deja correlacion ~0.999 dentro del trio
SEMILLAS_D2 = [1, 7, 42, 100, 2024]

def gemelas_de_lcavol(semilla):
    """Anade al train dos replicas ruidosas de `lcavol` (ya estandarizado, convencion ESL)."""
    rng = np.random.default_rng(semilla)
    copia1 = Xtr[:, 0] + rng.normal(0, SD_RUIDO_D2, Xtr.shape[0])
    copia2 = Xtr[:, 0] + rng.normal(0, SD_RUIDO_D2, Xtr.shape[0])
    return np.column_stack([Xtr, copia1, copia2])

filas_d2, correlaciones_d2 = [], []
for semilla in SEMILLAS_D2:
    Xg = gemelas_de_lcavol(semilla)
    c = np.corrcoef(Xg[:, [0, 8, 9]].T)
    correlaciones_d2 += [c[0, 1], c[0, 2], c[1, 2]]
    lasso_g = Lasso(alpha=0.05, max_iter=200000).fit(Xg, ytr)
    enet_g = ElasticNet(alpha=0.05, l1_ratio=0.3, max_iter=200000).fit(Xg, ytr)
    filas_d2.append(["LASSO", semilla] + [round(float(v), 3) for v in lasso_g.coef_[[0, 8, 9]]])
    filas_d2.append(["ElasticNet", semilla] + [round(float(v), 3) for v in enet_g.coef_[[0, 8, 9]]])

tabla_drill2 = pd.DataFrame(filas_d2, columns=["modelo", "semilla", "lcavol", "lcavol_copy1", "lcavol_copy2"])
corr_media_d2 = float(np.mean(correlaciones_d2))
print(tabla_drill2.to_string(index=False))
print(f"\nCorrelacion media dentro del trio {{lcavol, lcavol_copy1, lcavol_copy2}} = {corr_media_d2:.4f}")
print("LASSO (alpha=0.05): se queda con UNA de las tres y anula las otras, y cual conserva depende de la semilla.")
print("Elastic Net (alpha=0.05, l1_ratio=0.3): reparte el peso entre las tres en TODAS las semillas.")

**📖 Cómo se lee.** Las tres variables del trío correlacionan **≈ 0.999**. El **LASSO** conserva una y anula las otras dos, y **cuál** conserva cambia con la semilla del ruido (semillas 1 y 2024 → `lcavol`; semilla 7 → `copy1`; semilla 100 → `copy2`; semilla 42 reparte 0.094/0.454 entre las dos copias): la selección es una **lotería** entre gemelas. El **Elastic Net** reparte el peso entre las tres en las cinco semillas (**efecto de agrupamiento**, aportado por la componente L2). **⚠️** Concluir «solo un sensor aporta información, los otros dos se pueden retirar» a partir del LASSO es un error de lectura con costo real: si se retira el que sobrevivió por azar, se pierde la señal del grupo entero.

**🔎 Qué hace este código.** Arma la tabla comparativa OLS / RIDGE / LASSO de coeficientes estandarizados y MSE de prueba, para cotejarla con ESL Tabla 3.3.

In [ ]:
# Comparación con ESL Tabla 3.3 (coeficientes estandarizados)
tabla_esl = pd.DataFrame({
    "OLS": ols.coef_,
    "RIDGE": ridge.coef_,
    "LASSO (ESL)": lasso_esl.coef_,
}, index=predictores).round(3)
tabla_esl.loc["— MSE prueba —"] = [round(mse_test_ols, 3), round(mse_test_ridge, 3), round(mse_test_lasso, 3)]
print("Réplica de ESL Tabla 3.3 (predictores estandarizados):")
print(tabla_esl.to_string())
print("\nReferencia ESL: lcavol OLS 0.68, MSE OLS 0.521, RIDGE 0.496, LASSO 0.479 (3 vars).")

**Lectura de la réplica.** (1) `lcavol` es el coeficiente **mayor** en las tres columnas: el predictor dominante. (2) RIDGE contrae todo pero mantiene las 8 variables; el **LASSO anula 5** y deja un modelo de 3 variables {`lcavol`, `lweight`, `svi`}. (3) Regularizar **mejora la generalización**: LASSO ≤ RIDGE ≤ OLS en error de prueba. El LASSO logra el menor error **entre los cuatro modelos comparados** con el modelo más pequeño; ahora bien, las brechas de MSE (OLS 0.521 / RIDGE 0.497 / LASSO 0.485) son solo **≈ 0.13–0.20 del error estándar 0.179** → estadísticamente indistinguibles. La ventaja defendible del LASSO es la **parsimonia** (3 variables), no un mejor desempeño.

> **Nota técnica (honestidad de la réplica).** El λ exacto de ESL proviene de `glmnet` (R); scikit-learn (descenso por coordenadas) sitúa su λ_1se en un modelo algo menos ralo (≈5 variables). Se reproduce el modelo de 3 variables de ESL leyendo el *path* en su borde ralo. La **conclusión** es idéntica: unas pocas variables, encabezadas por `lcavol`/`lweight`/`svi`, explican el resultado. El coeficiente OLS de `lcavol` (0.676 vs 0.68) difiere por el denominador del escalado (`ddof=0` de sklearn frente a `ddof=1` de R): ~0.52 % (el escalador se ajusta sobre las 97 filas), irrelevante.

### 📄 En el paper — correspondencia con ESL Tabla 3.3 (p. 63)

| Término | LS (ESL) | venv OLS | Lasso (ESL) | venv Lasso |
|---|---:|---:|---:|---:|
| lcavol | 0.680 | 0.676 | 0.532 | 0.534 |
| lweight | 0.263 | 0.262 | 0.169 | 0.180 |
| svi | 0.305 | 0.304 | 0.092 | 0.080 |
| age, lbph, lcp, gleason, pgg45 | (varios) | (varios) | **.** (0) | **0** |
| **Test MSE** | **0.521** | **0.521** | **0.479** | **0.485** |

Fila a fila coincide con **ESL «Sección 3.4», Tabla 3.3, p. 63** y con Tibshirani (1996). El LASSO conserva `{lcavol, lweight, svi}` en ambas implementaciones. Los pequeños decimales difieren por `sklearn` vs `glmnet`/R (ver la ficha de la sesión de réplica del paper, «Sección 7»).

**🔎 Qué hace este código.** Separa, en **tres columnas con su fuente declarada**, lo que hasta ahora se publicaba en una sola bajo la etiqueta «ESL»: (1) los valores del **libro** ESL, Tabla 3.3, p. 63; (2) la **reproducción en R de T. Rigon**, que es de donde procedían realmente las cifras de Ridge y Lasso que el material llamaba «ESL»; y (3) el **venv** de este cuaderno. Lo que no se pudo verificar contra la fuente se deja **vacío**: no se rellena por simetría. Se vuelca a la hoja `benchmark_esl` del Excel.

In [ ]:
# REGISTRO: las TRES columnas del benchmark, cada una con su fuente (hoja `benchmark_esl`)
#  (1) ESL_libro  : Hastie, Tibshirani & Friedman, "The Elements of Statistical Learning", 2.a ed.,
#                   12.a impresion, Tabla 3.3, p. 63. Transcripcion verificada en
#                   los terminos que esa verificacion no cubre se dejan VACIOS, no se rellenan.
#  (2) Rigon_R    : reproduccion en R de T. Rigon (Data Mining, Univ. di Milano-Bicocca),
#                   https://tommasorigon.github.io/datamining/slides/un_C.html . Es la fuente REAL
#                   de las columnas Ridge y Lasso que el material venia etiquetando como "ESL".
#  (3) venv       : recomputado en este cuaderno con scikit-learn 1.6.1 (celdas de OLS, RIDGE y LASSO ralo).
VACIO = ""     # dato no verificado contra la fuente: se deja vacio a proposito

TERMINOS_BENCH = ["intercepto"] + predictores + ["test_error", "std_error", "n_no_nulos"]

esl_libro = {
    "OLS":   {"intercepto": VACIO, "lcavol": 0.680, "lweight": 0.263, "age": -0.141, "lbph": 0.210,
              "svi": 0.305, "lcp": -0.288, "gleason": -0.021, "pgg45": 0.267,
              "test_error": 0.521, "std_error": 0.179, "n_no_nulos": 8},
    "Ridge": {"intercepto": VACIO, "lcavol": 0.420, "lweight": 0.238, "age": -0.046, "lbph": 0.162,
              "svi": 0.227, "lcp": 0.000, "gleason": 0.040, "pgg45": 0.133,
              "test_error": 0.492, "std_error": 0.165, "n_no_nulos": 8},
    "Lasso": {"intercepto": VACIO, "lcavol": 0.533, "lweight": 0.169, "age": 0.000, "lbph": 0.002,
              "svi": 0.094, "lcp": 0.000, "gleason": 0.000, "pgg45": 0.000,
              "test_error": 0.479, "std_error": VACIO, "n_no_nulos": 4},
}
rigon_r = {
    "OLS":   {"intercepto": 2.465, "lcavol": 0.680, "lweight": 0.263, "age": -0.141, "lbph": 0.210,
              "svi": 0.305, "lcp": -0.288, "gleason": -0.021, "pgg45": 0.267,
              "test_error": 0.521, "std_error": 0.179, "n_no_nulos": 8},
    "Ridge": {"intercepto": 2.467, "lcavol": 0.588, "lweight": 0.258, "age": -0.113, "lbph": 0.201,
              "svi": 0.283, "lcp": -0.172, "gleason": 0.010, "pgg45": 0.204,
              "test_error": 0.496, "std_error": VACIO, "n_no_nulos": 8},
    "Lasso": {"intercepto": 2.468, "lcavol": 0.532, "lweight": 0.169, "age": 0.000, "lbph": 0.000,
              "svi": 0.092, "lcp": 0.000, "gleason": 0.000, "pgg45": 0.000,
              "test_error": 0.479, "std_error": VACIO, "n_no_nulos": 3},
}
_err2_ols = (yte - ols.predict(Xte)) ** 2
venv_bench = {}
for nombre, modelo, mse_modelo in [("OLS", ols, mse_test_ols), ("Ridge", ridge, mse_test_ridge),
                                   ("Lasso", lasso_esl, mse_test_lasso)]:
    fila = {"intercepto": round(float(modelo.intercept_), 4)}
    fila.update({p: round(float(c), 4) for p, c in zip(predictores, modelo.coef_)})
    fila["test_error"] = round(float(mse_modelo), 4)
    fila["std_error"] = round(float(_err2_ols.std(ddof=1) / np.sqrt(len(yte))), 4) if nombre == "OLS" else VACIO
    fila["n_no_nulos"] = int(np.sum(np.abs(modelo.coef_) > 1e-8))
    venv_bench[nombre] = fila

tabla_benchmark = pd.DataFrame(
    [[m, t, esl_libro[m][t], rigon_r[m][t], venv_bench[m][t]]
     for m in ["OLS", "Ridge", "Lasso"] for t in TERMINOS_BENCH],
    columns=["modelo", "termino", "ESL_libro", "Rigon_R", "venv"])

print(tabla_benchmark.to_string(index=False))
print("\nDiferencias que la etiqueta unica ocultaba:")
print(f"  Ridge lcavol      : ESL libro {esl_libro['Ridge']['lcavol']}  vs  Rigon {rigon_r['Ridge']['lcavol']}  vs  venv {venv_bench['Ridge']['lcavol']}")
print(f"  Ridge test_error  : ESL libro {esl_libro['Ridge']['test_error']}  vs  Rigon {rigon_r['Ridge']['test_error']}  vs  venv {venv_bench['Ridge']['test_error']}")
print(f"  Lasso lbph        : ESL libro {esl_libro['Lasso']['lbph']}  vs  Rigon {rigon_r['Lasso']['lbph']}  vs  venv {venv_bench['Lasso']['lbph']}")
print(f"  Lasso n_no_nulos  : ESL libro {esl_libro['Lasso']['n_no_nulos']}  vs  Rigon {rigon_r['Lasso']['n_no_nulos']}  vs  venv {venv_bench['Lasso']['n_no_nulos']}")

**📖 Cómo se lee.** La columna **LS (OLS)** sí coincide entre el libro y la reproducción de Rigon, y por eso la réplica nunca dio problemas ahí. Las columnas **Ridge** y **Lasso** **no** coinciden: el libro da Ridge `lcavol` **0.420** y error de prueba **0.492**, mientras que Rigon da **0.588** y **0.496**; y el Lasso del libro conserva `lbph = 0.002` —**4** coeficientes no nulos—, mientras que Rigon lo lleva a cero y deja **3**. **⚠️** Las cifras que el material publicaba como «ESL Tabla 3.3» para Ridge y Lasso son las de **Rigon**, no las del libro. La conclusión pedagógica no cambia (el LASSO da un modelo ralo con `{lcavol, lweight, svi}` y errores de prueba equivalentes), pero **la atribución sí**: cada columna se cita ahora por su fuente. Registro en la hoja `benchmark_esl`.

**🔎 Qué hace este código.** Calcula los *coefficient paths* de RIDGE y LASSO (coeficiente de cada variable frente a λ) y los guarda; los datos del path del LASSO se vuelcan luego al Excel para regenerar la figura desde ahí.

In [ ]:
# Coefficient paths: RIDGE y LASSO (se guardan datos y figura; los datos del LASSO
# se vuelcan luego al Excel para regenerar la figura desde ahí).
alphas_lasso, coefs_lasso, _ = lasso_path(Xtr, ytr, n_alphas=100)

alphas_ridge = np.logspace(-2, 4, 100)
coefs_ridge_path = np.array([Ridge(alpha=a).fit(Xtr, ytr).coef_ for a in alphas_ridge]).T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for j, nom in enumerate(predictores):
    axes[0].plot(alphas_ridge, coefs_ridge_path[j], color=PALETA[j % len(PALETA)], label=nom)
    axes[1].plot(alphas_lasso, coefs_lasso[j], color=PALETA[j % len(PALETA)], label=nom)
for ax, tit in zip(axes, ["RIDGE: nadie llega a cero", "LASSO: las variables se anulan una a una"]):
    ax.set_xscale("log"); ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("λ (escala log)"); ax.set_ylabel("Coeficiente estandarizado"); ax.set_title(tit)
axes[1].legend(fontsize=8, ncol=2, loc="upper right")
mostrar(fig, FIGURAS / "S05_fig_paths_ridge_lasso.png")

**Lectura del path.** En el LASSO, el **orden de salida es una jerarquía de importancia**: las variables que se anulan primero (con λ pequeño) son las menos útiles; `lcavol` resiste hasta el final → predictor más importante. En RIDGE las líneas se acercan a cero pero **ninguna lo toca**.

---

## 5.8 — ¿Qué decisión habilita? Laboratorio de negocio: Communities and Crime (alta dimensión) (Sección 5 del cuaderno)

**Dataset (Redmond, 2009, UCI id 183):** 1994 comunidades de EE. UU. descritas por variables socioeconómicas del censo de 1990; objetivo: `ViolentCrimesPerPop` (crímenes violentos por habitante, normalizado a [0,1]). Es el escenario de **alta dimensión** (~100 predictores, muchos correlacionados y con faltantes) donde la regularización destaca.

> **Buena práctica (contraste con el paso 4).** Aquí el escalado se ajusta **solo con el conjunto de entrenamiento** dentro de un `Pipeline` — evita la fuga de información. En la réplica de ESL se estandarizó sobre las 97 filas para reproducir su tabla; aquí prima la práctica de ML "pura".

**🔎 Qué hace este código.** Carga `communities.csv` (alta dimensión), descarta el bloque policial LEMAS (~84 % de faltantes) e imputa por la media las columnas con pocos faltantes.

In [ ]:
# Paso 1: cargar y manejar faltantes ('?' -> NaN)
crime = pd.read_csv(DATA / "communities.csv", na_values="?")
print("Dimensiones:", crime.shape)

no_predictivas = ["state", "county", "community", "communityname", "fold"]
objetivo = "ViolentCrimesPerPop"

# Descartar el bloque policial LEMAS (~84% faltante): imputarlo introduciria sesgo.
# OJO: 24 columnas superan el 50% de faltantes, pero DOS de ellas (`county` y `community`)
# son identificadores que ya estan en las no predictivas. El bloque LEMAS son las otras 22.
faltante = crime.isna().mean()
muy_faltantes = faltante[faltante > 0.5].index.tolist()
bloque_lemas = [c for c in muy_faltantes if c not in no_predictivas]
otras = [c for c in muy_faltantes if c in no_predictivas]
print(f"Columnas con >50% de faltantes: {len(muy_faltantes)}")
print(f"  de ellas, bloque LEMAS (encuesta policial, 84% faltante) que se descarta: {len(bloque_lemas)}")
print(f"  y {len(otras)} identificadores ya excluidos por no predictivos: {otras}")

Xc = crime.drop(columns=no_predictivas + [objetivo] + bloque_lemas, errors="ignore")
pocos_faltantes = [c for c in Xc.columns if Xc[c].isna().any()]
print("Columnas con pocos faltantes (se imputan por la media):", pocos_faltantes)
Xc = Xc.fillna(Xc.mean(numeric_only=True))
yc = crime[objetivo].values
print("Predictores finales:", Xc.shape[1])

**📖 Cómo se lee.** El dataset trae **1994 × 128**. Veinticuatro columnas superan el 50 % de faltantes, pero **dos de ellas —`county` (58.9 %) y `community` (59.0 %)— son identificadores** que ya estaban entre las 5 no predictivas: el **bloque LEMAS** que se descarta por faltante son las otras **22** (todas exactamente con 84 % de valores ausentes, porque solo 319 de las 1994 comunidades respondieron esa encuesta; imputarlas sesgaría). Queda **1 sola** columna con pocos faltantes (imputada por la media). La cuenta cierra: 128 − 5 no predictivas − 1 objetivo − 22 LEMAS = **100 predictores** socioeconómicos.

**❓ Qué se quiere averiguar.** Con 100 predictores y 1994 observaciones, ¿compensa el modelo complicado? Y si los cuatro presentan un error equivalente, ¿sobre qué se decide entonces?

- **Qué decide:** qué modelo se lleva a producción y, con él, cuántas variables hay que recolectar, documentar y explicar ante un regulador. Es el eje de la sesión: predecir mejor frente a interpretar mejor.
- **Antes de mirar el resultado:** si algún modelo bajara claramente el MSE de prueba, la elección sería técnica y no habría discusión. Si los cuatro quedan prácticamente empatados —alrededor de **0,0175**—, el error deja de discriminar y la única diferencia real es el número de variables activas: 100 en OLS y RIDGE frente a las que dejen LASSO y Elastic Net. Con un desempeño equivalente, la decisión es de negocio, y la regla de la sesión es elegir el modelo más simple y explicable.

**🔎 Qué hace este código.** Parte 70/30 y compara OLS / RIDGE / LASSO / Elastic Net, con el escalador ajustado **solo en train** dentro de un `Pipeline` (sin fuga de información). Reporta MSE de prueba y variables activas. **Sobre los pliegues de CV (`cv`):** el número de particiones con que se elige la penalización es un hiperparámetro de la **validación cruzada**, no del modelo. El cuaderno usa `cv=10` en `prostate` y en el `LassoCV` de este laboratorio, y `cv=5` en el `ElasticNetCV` de alta dimensión (n = 1994) solo para **acotar el cómputo** de la malla `l1_ratio`×`λ`; ambos son estándar y la **conclusión** (empate en MSE; LASSO 62 vs Elastic Net 72 variables) no depende del valor de `k`.

In [ ]:
# Paso 2: partición 70/30 y comparación OLS / RIDGE / LASSO / Elastic Net
# (escalado ajustado SOLO en train mediante Pipeline)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc.values, yc, test_size=0.3, random_state=RANDOM_STATE)

def evaluar(modelo):
    pipe = Pipeline([("escala", StandardScaler()), ("modelo", modelo)]).fit(Xc_tr, yc_tr)
    coef = pipe.named_steps["modelo"].coef_
    return (mean_squared_error(yc_te, pipe.predict(Xc_te)),
            int(np.sum(np.abs(coef) > 1e-8)), coef)

mse_ols_c,  n_ols_c,  _        = evaluar(LinearRegression())
mse_rid_c,  n_rid_c,  _        = evaluar(RidgeCV(alphas=np.logspace(-3, 3, 100)))
mse_las_c,  n_las_c,  coef_las = evaluar(LassoCV(cv=10, random_state=RANDOM_STATE, max_iter=100000, n_alphas=100))
mse_en_c,   n_en_c,   _        = evaluar(ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,1.0], cv=5,
                                                      random_state=RANDOM_STATE, max_iter=100000))

resumen_c = pd.DataFrame({
    "modelo": ["OLS", "RIDGE", "LASSO", "Elastic Net"],
    "MSE_prueba": [mse_ols_c, mse_rid_c, mse_las_c, mse_en_c],
    "variables_activas": [n_ols_c, n_rid_c, n_las_c, n_en_c],
})
print(resumen_c.round(5).to_string(index=False))

**📖 Cómo se lee.** Los cuatro modelos presentan un MSE de prueba prácticamente equivalente (**≈ 0.0175**): con n = 1994 y 100 predictores el OLS no sobreajusta gravemente. La diferencia es de **negocio**: el LASSO usa **62** variables y Elastic Net **72**, frente a las 100 del OLS/RIDGE.

**🔎 Qué hace este código.** Ordena las variables seleccionadas por el LASSO según el valor absoluto de su coeficiente y muestra las 12 más influyentes.

In [ ]:
# Paso 3: variables seleccionadas por el LASSO (jerarquía por |coeficiente|)
sel = pd.Series(coef_las, index=Xc.columns)
sel = sel[sel.abs() > 1e-8].reindex(sel.abs().sort_values(ascending=False).index)
print(f"El LASSO selecciona {n_las_c} de {Xc.shape[1]} predictores.")
print("\nTop 12 variables por |coeficiente|:")
print(sel.head(12).round(4).to_string())

**Recomendación (interpretabilidad vs. desempeño).** Los cuatro modelos alcanzan un MSE de prueba prácticamente idéntico: con `n = 1994` observaciones y ~100 predictores, el OLS no sobreajusta gravemente. La diferencia decisiva es de **negocio**: el LASSO consigue el mismo error usando **bastantes menos variables** y entrega una **lista ordenada e interpretable** (estructura familiar, pobreza, vivienda, composición demográfica). Cuando dos modelos presentan un error equivalente, se elige el **más simple y explicable** — especialmente en contextos regulados. Se recomienda el **LASSO** como modelo de trabajo y un **GAM** si se necesita capturar no linealidades manteniendo la interpretabilidad.

---

**🔎 Qué hace este código.** Repite la comparación de los cuatro modelos de Communities con **siete semillas** distintas (la semilla gobierna a la vez la partición 70/30 y la CV) y mide la **estabilidad del conjunto seleccionado** con el índice de Jaccard entre pares de semillas. Sirve para saber si «el LASSO deja 62 de 100» es una constante o una de las realizaciones posibles. Se vuelca a la hoja `estabilidad_semillas_S05` del Excel.

In [ ]:
# REGISTRO: cuanto se mueven las cifras de Communities al cambiar la semilla (hoja `estabilidad_semillas_S05`)
SEMILLAS_C = [42, 0, 1, 7, 13, 123, 2024]
COSTO_INDICADOR = 1200                 # costo ilustrativo por indicador y ano (misma unidad que la guia)
columnas_c = np.array(Xc.columns)

filas_c, seleccion_lasso, seleccion_enet = [], {}, {}
for semilla in SEMILLAS_C:
    Xa, Xb, ya, yb = train_test_split(Xc.values, yc, test_size=0.3, random_state=semilla)

    def evaluar_semilla(modelo):
        pipe = Pipeline([("escala", StandardScaler()), ("modelo", modelo)]).fit(Xa, ya)
        coef = pipe.named_steps["modelo"].coef_
        return float(mean_squared_error(yb, pipe.predict(Xb))), int(np.sum(np.abs(coef) > 1e-8)), coef

    m_ols, _, _ = evaluar_semilla(LinearRegression())
    m_rid, _, _ = evaluar_semilla(RidgeCV(alphas=np.logspace(-3, 3, 100)))
    m_las, n_las_s, c_las_s = evaluar_semilla(LassoCV(cv=10, random_state=semilla, max_iter=100000, n_alphas=100))
    m_ene, n_ene_s, c_ene_s = evaluar_semilla(ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, 1.0], cv=5,
                                                           random_state=semilla, max_iter=100000))
    seleccion_lasso[semilla] = set(columnas_c[np.abs(c_las_s) > 1e-8])
    seleccion_enet[semilla] = set(columnas_c[np.abs(c_ene_s) > 1e-8])
    filas_c.append([semilla, round(m_ols, 5), round(m_rid, 5), round(m_las, 5), round(m_ene, 5),
                    n_las_s, n_ene_s, COSTO_INDICADOR * (100 - n_las_s), COSTO_INDICADOR * (100 - n_ene_s)])

tabla_semillas = pd.DataFrame(filas_c, columns=["semilla", "mse_ols", "mse_ridge", "mse_lasso", "mse_enet",
                                                "n_lasso", "n_enet", "ahorro_lasso", "ahorro_enet"])

def jaccard_medio(seleccion):
    claves, valores = list(seleccion), []
    for i in range(len(claves)):
        for j in range(i + 1, len(claves)):
            a, b = seleccion[claves[i]], seleccion[claves[j]]
            valores.append(len(a & b) / len(a | b))
    return float(np.mean(valores))

jaccard_lasso = jaccard_medio(seleccion_lasso)
jaccard_enet = jaccard_medio(seleccion_enet)
nucleo_lasso = set.intersection(*seleccion_lasso.values())
nucleo_enet = set.intersection(*seleccion_enet.values())
sobreviven_de_42 = len(nucleo_lasso & seleccion_lasso[42])

print(tabla_semillas.to_string(index=False))
print(f"\nJaccard medio entre pares de semillas:  LASSO = {jaccard_lasso:.4f}   Elastic Net = {jaccard_enet:.4f}")
print(f"Variables que sobreviven en las {len(SEMILLAS_C)} semillas: LASSO {len(nucleo_lasso)} - Elastic Net {len(nucleo_enet)}")
print(f"De las {len(seleccion_lasso[42])} que selecciona la semilla 42, sobreviven en todas: {sobreviven_de_42}")
print(f"Rangos: LASSO {tabla_semillas.n_lasso.min()}-{tabla_semillas.n_lasso.max()} variables - "
      f"Elastic Net {tabla_semillas.n_enet.min()}-{tabla_semillas.n_enet.max()} - "
      f"MSE {tabla_semillas.mse_lasso.min():.5f}-{tabla_semillas.mse_lasso.max():.5f} - "
      f"ahorro {tabla_semillas.ahorro_lasso.min()}-{tabla_semillas.ahorro_lasso.max()}")

**📖 Cómo se lee.** Las cifras insignia del caso de negocio **no son constantes**: el LASSO conserva entre **44** (semilla 13) y **81** (semilla 7) de las 100 variables —62 con la semilla 42—, el Elastic Net entre **42** y **82**, y el MSE de prueba va de **0.01744** a **0.02043**. Traducido a dinero con el costo ilustrativo de 1 200 por indicador, el ahorro anual oscila entre **22 800** y **67 200**, frente a los 45 600 de la semilla 42. La estabilidad del **conjunto** también es limitada: el Jaccard medio entre pares de semillas es **0.67** (LASSO) y **0.69** (Elastic Net), y solo **33** de las 62 variables de la semilla 42 sobreviven en las siete. **⚠️** Lo defendible es el **orden de magnitud y la conclusión** —regularizar recorta un tercio o más de los indicadores sin perder error—, nunca el número exacto ni «este conjunto de 62 es estable». La ventaja de estabilidad del Elastic Net sobre el LASSO existe pero es **pequeña** (0.69 vs 0.67), no la que sugeriría el mecanismo por sí solo. Registro en la hoja `estabilidad_semillas_S05`.

## Transversal — Exportación a Excel y figuras de resultados (Sección 6 del cuaderno)

Convención del curso: los **resultados del modelo** se vuelcan a `resultados/S05_resultados.xlsx` (openpyxl) y las **figuras de resultados se generan leyendo ese Excel** (no desde objetos en memoria). Las figuras conceptuales (geometría L1 vs L2, curva de CV) se trazan por cálculo directo.

**🔎 Qué hace este código.** Vuelca al Excel de contrato `resultados/S05_resultados.xlsx` las **10 hojas** de la sesión: las **4 del contrato** —`regularizacion_prostate` (targets de réplica), `comparacion_modelos`, `paths` (camino del LASSO) y `communities`— y las **6 de registro de robustez** —`cv_barajado`, `benchmark_esl`, `estabilidad_semillas_S05`, `drill3_grado_semillas`, `drill2_gemelas` y `splines_naturales`—. **Es la única celda que escribe el Excel;** todas las figuras de resultados se generan leyéndolo, y el material de referencia de la sesión recomputa cada hoja desde los datos por una ruta independiente.

In [ ]:
# Construir el Excel: 4 hojas de CONTRATO + 6 hojas de REGISTRO DE ROBUSTEZ
from openpyxl import Workbook

wb = Workbook()

# Hoja 1: regularizacion_prostate (contrato exacto)
ws = wb.active
ws.title = "regularizacion_prostate"
ws["A1"] = "metrica"; ws["B1"] = "valor"
ws["A2"] = "coef_lcavol_ols";  ws["B2"] = float(round(coef_lcavol_ols, 4))
ws["A3"] = "lasso_n_nonzero";  ws["B3"] = int(lasso_n_nonzero)
ws["A4"] = "mse_test_ols";     ws["B4"] = float(round(mse_test_ols, 4))
ws["A5"] = "mse_test_lasso";   ws["B5"] = float(round(mse_test_lasso, 4))

# Hoja 2: comparacion_modelos (prostate)
ws2 = wb.create_sheet("comparacion_modelos")
ws2.append(["modelo", "mse_test", "n_coef_nonzero", "lambda", "alpha_l1_ratio"])
ws2.append(["OLS",         round(mse_test_ols, 4),   8,               "",                    ""])
ws2.append(["Ridge",       round(mse_test_ridge, 4), n_nz_ridge,      round(ridge.alpha_, 4), 0.0])
ws2.append(["Lasso",       round(mse_test_lasso, 4), lasso_n_nonzero, round(lambda_esl, 4),   1.0])
ws2.append(["ElasticNet",  round(mse_test_enet, 4),  n_nz_enet,       round(enet.alpha_, 4),  float(enet.l1_ratio_)])

# Hoja 3: paths (datos del coefficient path del LASSO para regenerar la figura)
ws3 = wb.create_sheet("paths")
ws3.append(["lambda"] + predictores)
for k in range(alphas_lasso.size):
    ws3.append([float(alphas_lasso[k])] + [float(coefs_lasso[j, k]) for j in range(len(predictores))])

# Hoja 4: communities (OLS vs LASSO — y los otros dos modelos)
ws4 = wb.create_sheet("communities")
ws4.append(["modelo", "mse_test", "n_variables_seleccionadas", "n_predictores_totales"])
ws4.append(["OLS",         round(mse_ols_c, 5), n_ols_c, Xc.shape[1]])
ws4.append(["Ridge",       round(mse_rid_c, 5), n_rid_c, Xc.shape[1]])
ws4.append(["LASSO",       round(mse_las_c, 5), n_las_c, Xc.shape[1]])
ws4.append(["ElasticNet",  round(mse_en_c, 5),  n_en_c,  Xc.shape[1]])

# --- Hojas de REGISTRO DE ROBUSTEZ (no forman parte del contrato de la replica) ---
def _celda(v):
    "Convierte escalares de numpy a tipos nativos para que openpyxl los escriba."
    return v.item() if hasattr(v, "item") else v

# Hoja 5: cv_barajado (CV sin barajar vs barajada, sobre un archivo ordenado por la respuesta)
ws5 = wb.create_sheet("cv_barajado")
ws5.append(["criterio", "cv_sin_barajar", "cv_barajado_semilla42"])
for clave in ["lambda_min", "n_vars_lambda_min", "cv_mse_lambda_min", "se_cv_lambda_min",
              "mse_test_lambda_min", "lambda_1se", "n_vars_lambda_1se", "cv_mse_lambda_1se",
              "mse_test_lambda_1se", "vars_lambda_1se", "enet_l1_ratio", "enet_lambda",
              "enet_n_vars", "enet_mse_test"]:
    a, b = registro_cv_sin[clave], registro_cv_con[clave]
    ws5.append([clave,
                round(a, 4) if isinstance(a, float) else a,
                round(b, 4) if isinstance(b, float) else b])
ws5.append(["corr_orden_filas_lpsa", round(corr_orden_lpsa, 4), round(corr_orden_lpsa, 4)])
ws5.append([])
ws5.append(["lambda_1se por semilla de barajado (KFold shuffle=True)"])
ws5.append(list(tabla_cv_semillas.columns))
for fila in tabla_cv_semillas.itertuples(index=False):
    ws5.append([_celda(v) for v in fila])

# Hoja 6: benchmark_esl (ESL libro | Rigon en R | venv, cada columna con su fuente)
ws6 = wb.create_sheet("benchmark_esl")
ws6.append(list(tabla_benchmark.columns))
for fila in tabla_benchmark.itertuples(index=False):
    ws6.append([_celda(v) for v in fila])
ws6.append([])
ws6.append(["fuente_ESL_libro",
            "Hastie, Tibshirani & Friedman, The Elements of Statistical Learning, 2.a ed., 12.a impresion, "
            "Tabla 3.3, p. 63. Transcripcion verificada en la verificación de la sesión (15/08/2026). "
            "El PDF no esta en papers/: las celdas vacias son terminos que esa verificacion no cubre."])
ws6.append(["fuente_Rigon_R",
            "T. Rigon, Data Mining (Univ. degli Studi di Milano-Bicocca), reproduccion en R: "
            "https://tommasorigon.github.io/datamining/slides/un_C.html . Es la fuente real de las columnas "
            "Ridge y Lasso que el material publicaba bajo la etiqueta 'ESL Tabla 3.3'."])
ws6.append(["fuente_venv",
            "Recomputado en notebook/S05_regularizacion.ipynb con scikit-learn 1.6.1 sobre data/prostate.csv "
            "(convencion ESL: escalador ajustado sobre las 97 filas; LASSO en el borde ralo, lambda 0.2203)."])

# Hoja 7: estabilidad_semillas_S05 (Communities con 7 semillas + Jaccard de la seleccion)
ws7 = wb.create_sheet("estabilidad_semillas_S05")
ws7.append(list(tabla_semillas.columns))
for fila in tabla_semillas.itertuples(index=False):
    ws7.append([_celda(v) for v in fila])
ws7.append([])
ws7.append(["jaccard_medio_lasso", round(jaccard_lasso, 4)])
ws7.append(["jaccard_medio_elastic_net", round(jaccard_enet, 4)])
ws7.append(["nucleo_comun_lasso_7_semillas", len(nucleo_lasso)])
ws7.append(["nucleo_comun_enet_7_semillas", len(nucleo_enet)])
ws7.append(["de_las_seleccionadas_por_la_semilla_42_sobreviven_en_las_7", sobreviven_de_42])
ws7.append(["rango_n_lasso", f"{int(tabla_semillas.n_lasso.min())}-{int(tabla_semillas.n_lasso.max())}"])
ws7.append(["rango_n_enet", f"{int(tabla_semillas.n_enet.min())}-{int(tabla_semillas.n_enet.max())}"])
ws7.append(["rango_mse_lasso", f"{tabla_semillas.mse_lasso.min():.5f}-{tabla_semillas.mse_lasso.max():.5f}"])
ws7.append(["rango_ahorro_lasso", f"{int(tabla_semillas.ahorro_lasso.min())}-{int(tabla_semillas.ahorro_lasso.max())}"])
ws7.append(["costo_ilustrativo_por_indicador", COSTO_INDICADOR])

# Hoja 8: drill3_grado_semillas (grado del polinomio por CV, semilla a semilla)
ws8 = wb.create_sheet("drill3_grado_semillas")
ws8.append(list(tabla_drill3.columns))
for fila in tabla_drill3.itertuples(index=False):
    ws8.append([_celda(v) for v in fila])
ws8.append([])
ws8.append(["se_del_argmin_semilla42", round(se_argmin_42, 4)])
ws8.append(["techo_1_error_estandar", round(techo_1se_d3, 4)])
ws8.append(["grados_dentro_de_1_se", " ".join(str(g) for g in grados_dentro_1se)])
ws8.append(["argmin_por_semilla", " ".join(f"{s}:{g}" for s, g in zip(tabla_drill3['semilla'], tabla_drill3['argmin']))])

# Hoja 9: drill2_gemelas (construccion reproducible del Drill 2)
ws9 = wb.create_sheet("drill2_gemelas")
ws9.append(list(tabla_drill2.columns))
for fila in tabla_drill2.itertuples(index=False):
    ws9.append([_celda(v) for v in fila])
ws9.append([])
ws9.append(["sd_del_ruido", SD_RUIDO_D2])
ws9.append(["generador", "np.random.default_rng(semilla); dos extracciones normal(0, sd) consecutivas"])
ws9.append(["base", "train de 67 filas, 8 predictores estandarizados con la convencion ESL (97 filas)"])
ws9.append(["corr_media_del_trio", round(corr_media_d2, 4)])
ws9.append(["alpha_lasso", 0.05])
ws9.append(["alpha_y_l1_ratio_elastic_net", "0.05 / 0.3"])

# Hoja 10: splines_naturales (la base que el silabo prescribe y el cuaderno no ajustaba)
ws10 = wb.create_sheet("splines_naturales")
ws10.append(list(tabla_splines.columns))
for fila in tabla_splines.itertuples(index=False):
    ws10.append([_celda(v) for v in fila])

wb.save(XLSX)
print("Excel guardado en:", XLSX)
print("Hojas:", wb.sheetnames)

**📖 Cómo se lee.** El Excel queda con **10 hojas**. La hoja `regularizacion_prostate` fija el contrato de la réplica (coeficiente de `lcavol`, nº de variables del LASSO y los dos MSE de prueba) y es la que leen el validador, el deck y la evaluación; `comparacion_modelos`, `paths` y `communities` sostienen las figuras de resultados. Las seis hojas de registro no forman parte del contrato de la réplica: documentan **cuánto se mueve cada cifra** al cambiar la partición, la semilla o la fuente del benchmark, y existen para que ninguna cifra publicada como constante lo sea sin haberse medido.

### ✅ Verificación desde la base — transversal (subsección 6.1) <a id="verif-base"></a>

Antes de que el deck y la evaluación confíen en el Excel, se comprueba que ese registro es **producto de ejecutar el código sobre la base**, no un valor tecleado. Se **recomputa** desde el `prostate` ya cargado —de forma independiente de los objetos ya ajustados— el coeficiente OLS de `lcavol`, el MSE de prueba del OLS y el del LASSO ralo, y se cruza con las celdas del Excel mediante `assert`. Refleja, visible y explicada, la lógica de el material de referencia de la sesión.

**🔎 Qué hace este código.** Reajusta OLS y LASSO desde `prost` (independiente de los modelos ya entrenados), lee las celdas B2–B5 del Excel y comprueba con `assert` que **recomputado ≈ paper ≈ Excel**. No escribe en el Excel.

In [ ]:
# ✅ recomputar desde la base y cruzar recomputado ~ paper ~ Excel (NO escribe en el Excel)
from openpyxl import load_workbook

Xv = StandardScaler().fit(prost[predictores].values).transform(prost[predictores].values)  # convencion ESL (97 filas)
trv = (prost["train"] == "T").values; tev = ~trv
yv = prost["lpsa"].values
ols_v = LinearRegression().fit(Xv[trv], yv[trv])
coef_lcavol_v = ols_v.coef_[predictores.index("lcavol")]
mse_ols_v = mean_squared_error(yv[tev], ols_v.predict(Xv[tev]))
# LASSO en el borde ralo {lcavol, lweight, svi} (misma logica que el Paso 5b)
lam_v = next(a for a in np.logspace(np.log10(0.05), np.log10(0.5), 400)
             if set(np.array(predictores)[np.abs(Lasso(alpha=a, max_iter=200000).fit(Xv[trv], yv[trv]).coef_) > 1e-8])
             == {"lcavol", "lweight", "svi"})
lasso_v = Lasso(alpha=lam_v, max_iter=200000).fit(Xv[trv], yv[trv])
mse_lasso_v = mean_squared_error(yv[tev], lasso_v.predict(Xv[tev]))
n_nz_v = int(np.sum(np.abs(lasso_v.coef_) > 1e-8))

ws_v = load_workbook(XLSX, data_only=True)["regularizacion_prostate"]
excel = {ws_v[f"A{r}"].value: ws_v[f"B{r}"].value for r in range(2, 6)}

tabla_verif = pd.DataFrame({
    "recomputado (venv)": [round(coef_lcavol_v, 4), n_nz_v, round(mse_ols_v, 4), round(mse_lasso_v, 4)],
    "ESL / paper":        [0.68, "3-4", 0.521, 0.479],
    "Excel (contrato)":   [excel["coef_lcavol_ols"], excel["lasso_n_nonzero"], excel["mse_test_ols"], excel["mse_test_lasso"]],
}, index=["coef_lcavol_ols", "lasso_n_nonzero", "mse_test_ols", "mse_test_lasso"])
print(tabla_verif.to_string())

assert abs(coef_lcavol_v - excel["coef_lcavol_ols"]) < 1e-3
assert abs(mse_ols_v - excel["mse_test_ols"]) < 1e-3
assert abs(mse_lasso_v - excel["mse_test_lasso"]) < 1e-3
assert n_nz_v == excel["lasso_n_nonzero"]
assert abs(coef_lcavol_v - 0.68) <= 0.03 and abs(mse_ols_v - 0.521) <= 0.03   # dentro de tolerancia del paper
print("\nOK: recomputado desde la base ~ paper (tolerancia) y ~ Excel (contrato).")

**📖 Cómo se lee.** Las tres columnas coinciden: lo **recomputado** desde `prostate` reproduce el **paper** dentro de tolerancia (`lcavol` 0.676 vs 0.68; MSE OLS 0.5213 vs 0.521) y coincide con el **Excel** hasta el cuarto decimal. El valor operativo (venv, 0.676) manda; el del paper (0.68) es el *benchmark* etiquetado. Los `assert` fallarían si alguien editara el Excel manualmente o si la base cambiara: por eso el registro es *auditable*. La versión ejecutable con red vive en el material de referencia de la sesión.

### Figuras de resultados (leídas del Excel) y figuras conceptuales — transversal (subsección 6.2)

**🔎 Qué hace este código.** Lee la hoja `comparacion_modelos` del Excel y grafica el MSE de prueba de los 4 modelos sobre `prostate` (figura de resultados generada **desde el Excel**, no desde objetos en memoria).

In [ ]:
# FIGURA DE RESULTADOS 1 (leyendo el Excel): comparación de MSE de prueba de los 4 modelos
comp = pd.read_excel(XLSX, sheet_name="comparacion_modelos")
fig, ax = plt.subplots(figsize=(6.6, 4))
barras = ax.bar(comp["modelo"], comp["mse_test"], color=[UPC_TINTA, UPC_GRIS, UPC_ROJO, "#E4879C"])
for b, v in zip(barras, comp["mse_test"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.003, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("MSE de prueba")
ax.set_title("Regularización vs. OLS en `prostate` (menor es mejor)")
ax.set_ylim(0, max(comp["mse_test"]) * 1.18)
mostrar(fig, FIGURAS / "S05_fig_comparacion_mse_prostate.png")

**🔎 Qué hace este código.** Lee la hoja `paths` del Excel y reconstruye los *coefficient paths* del LASSO en `prostate`.

In [ ]:
# FIGURA DE RESULTADOS 2 (leyendo el Excel): coefficient paths del LASSO
paths = pd.read_excel(XLSX, sheet_name="paths")
fig, ax = plt.subplots(figsize=(7, 4.6))
for j, nom in enumerate(predictores):
    ax.plot(paths["lambda"], paths[nom], color=PALETA[j % len(PALETA)], label=nom)
ax.set_xscale("log"); ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("λ (escala log)"); ax.set_ylabel("Coeficiente estandarizado")
ax.set_title("Coefficient paths del LASSO en `prostate`\n(reconstruido desde el Excel de resultados)")
ax.legend(fontsize=8, ncol=2)
mostrar(fig, FIGURAS / "S05_fig_lasso_paths_prostate.png")

**🔎 Qué hace este código.** Lee la hoja `communities` del Excel y compara los 4 modelos en alta dimensión: MSE de prueba (empate) y nº de variables seleccionadas.

In [ ]:
# FIGURA DE RESULTADOS 3 (leyendo el Excel): comparación en Communities
comm = pd.read_excel(XLSX, sheet_name="communities")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.bar(comm["modelo"], comm["mse_test"], color=[UPC_TINTA, UPC_GRIS, UPC_ROJO, "#E4879C"])
a1.set_ylabel("MSE de prueba"); a1.set_title("MSE de prueba (empate práctico)")
a1.tick_params(axis="x", rotation=15)
a2.bar(comm["modelo"], comm["n_variables_seleccionadas"], color=[UPC_TINTA, UPC_GRIS, UPC_ROJO, "#E4879C"])
a2.axhline(comm["n_predictores_totales"].iloc[0], color="k", ls="--",
           label=f"total = {int(comm['n_predictores_totales'].iloc[0])}")
a2.set_ylabel("Variables activas"); a2.set_title("El LASSO usa muchas menos variables")
a2.legend(); a2.tick_params(axis="x", rotation=15)
mostrar(fig, FIGURAS / "S05_fig_communities_comparacion.png")

**🔎 Qué hace este código.** Figura **conceptual** (cálculo directo, no lee el Excel): la geometría de la restricción L1 (rombo) frente a L2 (círculo) y por qué el contacto con las curvas de nivel del RSS ocurre en un vértice del rombo (coeficiente = 0).

In [ ]:
# FIGURA CONCEPTUAL 1 (cálculo directo): geometría L1 (rombo) vs L2 (círculo)
theta = np.linspace(0, 2*np.pi, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
b0 = np.array([1.9, 1.3])                    # solución OLS (centro de las elipses del RSS)
for ax, tipo in zip(axes, ["L2 (RIDGE): círculo", "L1 (LASSO): rombo"]):
    for r in [0.5, 1.0, 1.6, 2.3]:           # curvas de nivel del RSS (elipses)
        ax.plot(b0[0] + r*1.5*np.cos(theta), b0[1] + r*np.sin(theta), color=UPC_GRIS, lw=0.8)
    if tipo.startswith("L2"):
        ax.plot(np.cos(theta), np.sin(theta), color=UPC_ROJO, lw=2)
        ax.plot(0.78, 0.63, "o", color=UPC_TINTA, ms=9)      # contacto fuera de los ejes
    else:
        ax.plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], color=UPC_ROJO, lw=2)
        ax.plot(0, 1, "o", color=UPC_TINTA, ms=9)            # contacto en un vértice: β1 = 0
        ax.annotate("vértice → un\ncoeficiente = 0", (0, 1), (-1.9, 1.5), fontsize=9,
                    arrowprops=dict(arrowstyle="->", color=UPC_TINTA))
    ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
    ax.set_xlim(-2.2, 3.2); ax.set_ylim(-1.8, 2.4); ax.set_aspect("equal")
    ax.set_xlabel("β₁"); ax.set_ylabel("β₂"); ax.set_title(tipo); ax.grid(False)
fig.suptitle("Por qué el LASSO anula coeficientes y el RIDGE no", y=1.02)
mostrar(fig, FIGURAS / "S05_fig_geometria_L1_L2.png")

**Lectura.** El LASSO restringe los coeficientes a un **rombo** con vértices sobre los ejes; el contacto con las curvas de nivel del RSS ocurre con alta probabilidad en un **vértice**, donde una coordenada vale exactamente 0 → variable eliminada. El RIDGE restringe a un **círculo** sin esquinas: el contacto casi nunca cae sobre un eje → los coeficientes se contraen pero no se anulan.

**🔎 Qué hace este código.** Figura **conceptual** (cálculo directo): la curva de validación cruzada del LASSO en `prostate`, con la banda de ±1 se (se=sd/√k) y las marcas de λ_min y λ_1se.

In [ ]:
# FIGURA CONCEPTUAL 2 (cálculo directo): curva de validación cruzada del LASSO en `prostate`
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(lasso_cv.alphas_, mse_media, color=UPC_ROJO, label="MSE de CV")
ax.fill_between(lasso_cv.alphas_, mse_media - mse_se, mse_media + mse_se,
                color=UPC_ROJO, alpha=0.15, label="± 1 error estándar (se=sd/√k)")
ax.axvline(lasso_cv.alpha_, color=UPC_TINTA, ls="--", label=f"λ_min = {lasso_cv.alpha_:.3f}")
ax.axvline(lambda_1se, color=UPC_GRIS, ls=":", label=f"λ_1se = {lambda_1se:.3f}")
ax.set_xscale("log")
ax.set_xlabel("λ (escala log)"); ax.set_ylabel("MSE de validación cruzada")
ax.set_title("Elección de λ por CV: λ_min (desempeño) vs. λ_1se (parsimonia)")
ax.legend(fontsize=9)
mostrar(fig, FIGURAS / "S05_fig_cv_lambda_lasso.png")

**Lectura.** La curva de CV tiene forma de U: error alto con λ muy bajo (sobreajuste) y con λ muy alto (subajuste). **λ_min** da el menor error (máximo desempeño); **λ_1se** elige el modelo más simple cuyo error sigue dentro de un error estándar (se=sd/√k) del mínimo. Elegir entre ambos es una **decisión de negocio**: desempeño puro frente a parsimonia e interpretabilidad.

---

## 5.3 en profundidad — Construye el pipeline de regularización desde cero (Sección 7 del cuaderno)  <a id="sec-pipeline"></a>

El alumno **rearma el flujo completo sin los objetos ya ajustados**: del CSV crudo a la decisión, pasando por estandarización, elección de λ y selección de variables. El objetivo es reproducir, de forma independiente, el **mismo modelo LASSO ralo** del contrato (3 variables y su MSE de prueba). Si el resultado coincide (con `assert`), queda demostrado que **es el método —no un atajo— el que produce la réplica**.

**🔎 Qué hace este código.** Lee `prostate.csv` desde cero, estandariza los 8 predictores con la **convención de ESL** (escalador ajustado sobre las 97 filas), ajusta el OLS y el LASSO en el borde ralo `{lcavol, lweight, svi}`, y **comprueba con `assert`** que el coeficiente de `lcavol`, el nº de variables activas y los MSE de prueba coinciden con el Excel de contrato. No usa `prost`, `escala_esl` ni ningún objeto previo; tampoco reescribe el Excel.

In [ ]:
# 🧱 pipeline regularizado reconstruido desde el CSV crudo (sin helpers; NO escribe en el Excel)
from openpyxl import load_workbook

raw = pd.read_csv(DATA / "prostate.csv")                        # 1) base cruda
P = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
m_tr = (raw["train"].astype(str).str.upper() == "T").values     # particion fija de ESL
# 2) estandarizar (convencion ESL: sobre las 97 filas -> reproduce la Tabla 3.3)
esc = StandardScaler().fit(raw[P].values)
Xz = esc.transform(raw[P].values); yz = raw["lpsa"].values
Xz_tr, Xz_te, yz_tr, yz_te = Xz[m_tr], Xz[~m_tr], yz[m_tr], yz[~m_tr]
# 3) OLS
ols_s = LinearRegression().fit(Xz_tr, yz_tr)
coef_lcavol_s = ols_s.coef_[P.index("lcavol")]
mse_ols_s = mean_squared_error(yz_te, ols_s.predict(Xz_te))
# 4) LASSO: elegir lambda en el borde ralo {lcavol, lweight, svi} y ajustar
lam_s = next(a for a in np.logspace(np.log10(0.05), np.log10(0.5), 400)
             if set(np.array(P)[np.abs(Lasso(alpha=a, max_iter=200000).fit(Xz_tr, yz_tr).coef_) > 1e-8])
             == {"lcavol", "lweight", "svi"})
lasso_s = Lasso(alpha=lam_s, max_iter=200000).fit(Xz_tr, yz_tr)
sel_s = list(np.array(P)[np.abs(lasso_s.coef_) > 1e-8])
mse_lasso_s = mean_squared_error(yz_te, lasso_s.predict(Xz_te))
print(f"coef lcavol (OLS) = {coef_lcavol_s:.4f}   MSE test OLS = {mse_ols_s:.4f}")
print(f"LASSO lambda = {lam_s:.4f}   variables ({len(sel_s)}): {sel_s}   MSE test = {mse_lasso_s:.4f}")

# 5) comprobar contra el Excel de contrato
c = load_workbook(XLSX, data_only=True)["regularizacion_prostate"]
cc = {c[f"A{r}"].value: c[f"B{r}"].value for r in range(2, 6)}
assert abs(coef_lcavol_s - cc["coef_lcavol_ols"]) < 1e-3
assert len(sel_s) == cc["lasso_n_nonzero"] and set(sel_s) == {"lcavol", "lweight", "svi"}
assert abs(mse_ols_s - cc["mse_test_ols"]) < 1e-3
assert abs(mse_lasso_s - cc["mse_test_lasso"]) < 1e-3
print("\nOK: el pipeline desde cero reproduce el contrato (coef 0.676 | 3 variables | MSE OLS 0.5213 | MSE LASSO 0.4851).")

**📖 Cómo se lee.** Partiendo del CSV crudo y sin reutilizar ningún objeto, el pipeline reproduce el contrato exacto: `lcavol` = 0.676, LASSO de **3 variables** `{lcavol, lweight, svi}`, MSE de prueba OLS 0.5213 y LASSO 0.4851. La réplica no dependía de un estado oculto del cuaderno: el **método** la produce. **⚠️ Fuga leve, declarada:** este atajo **hornea la respuesta** en la búsqueda de λ —recorre el *path* hasta que el conjunto activo es exactamente `{lcavol, lweight, svi}`, el resultado que ya se conoce de ESL—, de modo que presupone la selección en lugar de descubrirla. Sirve para fijar el λ operativo de la réplica; el flujo que **no** presupone la respuesta es elegir λ por `LassoCV` (λ_min/λ_1se), como se propone justo abajo.

**✍️ Ahora, por cuenta propia.** Se propone repetir el pipeline (a) estandarizando **solo con el train** —observar que `lcavol` pasa a ≈ 0.711 mientras el MSE de prueba del OLS no cambia (ver «Sección 2.1» de la guía de supuestos de la sesión)—; (b) eligiendo λ por `LassoCV` en vez del borde ralo; y (c) sustituyendo el LASSO por Elastic Net y comparando la lista seleccionada.

## 5.6 — ¿Cuándo se puede confiar en un modelo regularizado? Supuestos: cómo identificarlos y corregirlos (Sección 8 del cuaderno) <a id="supuestos"></a>

> **Fuente canónica:** la guía de supuestos de la sesión (qué es, cómo se identifica, cómo se corrige por método, alcance). Aquí se ejecutan los **diagnósticos**; su desarrollo teórico y sus fuentes viven en ese documento. **Ninguna celda de esta sección escribe en el Excel de contrato.**
>
> **Regla de alcance de S05.** El trabajo sobre cada supuesto llega hasta **(a) diagnosticarlo** —con su prueba, gráfico o señal y un umbral práctico— y **(b) aplicar o nombrar la corrección propia de la sesión** (regularización L1/L2, estandarización, elección de λ/α por CV, modelado no lineal). Lo que pertenece a otras sesiones (PCA → S06; GLS/series → S11) o a inferencia más fina se **nombra**, no se ejecuta.

**Supuestos del modelo lineal que la regularización mitiga o hereda** (`SUPUESTOS_S05.md`, Parte 1):

| Supuesto | Cómo identificar | Cómo corregir (método) | Alcance |
|---|---|---|---|
| **1.1 Linealidad** | Residuos vs. ajustados (curva); mejora del error de CV al añadir curvatura | Polinomios / splines / GAM (parte no lineal) | **S05** |
| **1.2 Sin multicolinealidad severa** | VIF (>5 preocupa, >10 severa); correlaciones >0,8–0,9; coeficientes inestables | **RIDGE** (robustez a colinealidad); Elastic Net — *PCA → S06* | **S05 — central** |
| **1.3 Homocedasticidad** | Residuos vs. ajustados (embudo) | Transformar `Y` (la réplica usa `log(psa)`) — ES robustos (avanzado) | Diagnóstico S05 |
| **1.4 Normalidad de errores** | Q-Q plot de residuos | Transformar `Y`; TCL en muestra grande — *inferencia post-LASSO (avanzado)* | S05 (coef. regularizado es predictivo) |
| **1.5 Independencia** | Revisar el diseño (temporal/agrupado) | CV por grupos/bloques (`GroupKFold`,`TimeSeriesSplit`) — *series → S11* | Diagnóstico S05 |

**Requisitos propios de la regularización y de la parte no lineal** (`SUPUESTOS_S05.md`, Partes 2 y 3):

| Requisito | Cómo identificar | Cómo corregir | Alcance |
|---|---|---|---|
| **2.1 Estandarización obligatoria** | Selección dominada por la escala; media/var del train escalado ≠ 0/1 | `StandardScaler` en `Pipeline`, ajuste **solo en train** | **S05 — imprescindible** |
| **2.2 λ/α por CV** | Curva de CV en U con banda ±1 se | `RidgeCV`/`LassoCV`/`ElasticNetCV`; λ_min vs. λ_1se | **S05** |
| **2.3 Sesgo-varianza** | Brecha entrenamiento–prueba; curvas de validación y de aprendizaje | Ajustar λ/complejidad por CV; más datos | **S05 — central** |
| **2.4 Selección del LASSO (correlación → inestabilidad)** | Reejecutar cambia la selección; path; correlaciones altas | **Elastic Net** (efecto de agrupamiento) | **S05** |
| **3.1 Grado del polinomio / extrapolación** | Grado por CV (U); oscilaciones en las colas | Grado por CV; **splines naturales**; no extrapolar | **S05** |

**🔎 Qué hace este código.** Diagnostica la **multicolinealidad** (supuesto 1.2) sobre los 8 predictores de `prostate`: calcula el **VIF** de cada uno y la **matriz de correlaciones**, y dibuja el heatmap. Umbral práctico: VIF > 5 preocupa, > 10 severa; correlaciones absolutas > 0,8–0,9 señalan redundancia.

In [ ]:
# Diagnostico 1.2 - multicolinealidad: VIF + matriz de correlaciones (NO escribe en el Excel)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

Xc_vif = sm.add_constant(pd.DataFrame(X, columns=predictores))   # X = predictores estandarizados (97 filas)
vif = pd.Series({predictores[j]: variance_inflation_factor(Xc_vif.values, j + 1) for j in range(len(predictores))})
print("VIF por predictor (umbral: >5 preocupa, >10 severa):")
print(vif.round(2).to_string())
print(f"\nVIF maximo = {vif.max():.2f} ({vif.idxmax()})  ->  {'sin multicolinealidad severa' if vif.max() < 5 else 'REVISAR'}")

corr = pd.DataFrame(prost[predictores].values, columns=predictores).corr()
pares = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
         .abs().sort_values(ascending=False))
print("\nPares de predictores mas correlacionados:")
print(pares.head(3).round(3).to_string())

fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(predictores))); ax.set_xticklabels(predictores, rotation=45, ha="right")
ax.set_yticks(range(len(predictores))); ax.set_yticklabels(predictores)
for i in range(len(predictores)):
    for j in range(len(predictores)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title("Correlacion entre predictores de prostate")
mostrar(fig, FIGURAS / "S05_sup_correlacion.png")

**📖 Cómo se lee.** El VIF máximo es **≈ 3.1** (`lcp`) y todos quedan **por debajo de 5**: `prostate` **no** sufre multicolinealidad severa (por eso RIDGE y OLS dan coeficientes parecidos aquí). El par más correlacionado es **`gleason`–`pgg45` (≈ 0.75)**, seguido de `lcavol`–`lcp` y `svi`–`lcp`. **💡** Donde el VIF SÍ aumenta de forma pronunciada (p. ej. Communities, «Sección 5»), RIDGE/Elastic Net son la corrección: estabilizan los coeficientes al contraerlos. Método y umbrales en la guía de supuestos de la sesión («Sección 1.2»).

**🔎 Qué hace este código.** Verifica el requisito 2.1: tras estandarizar con el escalador de la réplica, comprueba que las **medias del train escalado ≈ 0** y las **varianzas ≈ 1**, y compara con el ajuste del escalador **solo con el train** (flujo de ML «puro»).

In [ ]:
# Diagnostico 2.1 - verificacion de la estandarizacion (NO escribe en el Excel)
print("Convencion ESL (escalador ajustado sobre las 97 filas, para reproducir la Tabla 3.3):")
print(f"  media de X_train escalado = {Xtr.mean():+.3f}   varianza = {Xtr.var():.3f}")
esc_tr = StandardScaler().fit(prost.loc[tr, predictores].values)    # ajuste SOLO con el train (flujo ML puro)
Xtr_only = esc_tr.transform(prost.loc[tr, predictores].values)
Xte_only = esc_tr.transform(prost.loc[te, predictores].values)
print("Flujo ML (escalador ajustado solo con el train):")
print(f"  media de X_train escalado = {Xtr_only.mean():+.3f}   varianza = {Xtr_only.var():.3f}")

# Dos cantidades DISTINTAS que no deben confundirse:
factor_ddof = float(np.sqrt(96 / 97))                       # (a) sklearn ddof=0 vs R ddof=1, misma convencion
efecto_ddof = 100 * (1 - factor_ddof)
ols_esl = LinearRegression().fit(Xtr, ytr)                  # (b) cambiar de convencion de estandarizacion
ols_solo_train = LinearRegression().fit(Xtr_only, ytr)
coef_esl = float(ols_esl.coef_[predictores.index("lcavol")])
coef_solo_train = float(ols_solo_train.coef_[predictores.index("lcavol")])
efecto_convencion = 100 * (coef_solo_train / coef_esl - 1)

print("\nAmbos criterios dejan el train en media ~0 y varianza ~1. Dos efectos que NO son el mismo:")
print(f"  (a) criterio ddof (poblacional sklearn vs muestral de R), con la MISMA convencion de 97 filas:")
print(f"      factor sqrt(96/97) = {factor_ddof:.6f}  ->  {efecto_ddof:.3f} % sobre los coeficientes")
print(f"  (b) cambiar de convencion (ESL 97 filas -> escalador solo-train):")
print(f"      coef lcavol {coef_esl:.4f} -> {coef_solo_train:.4f}  =  {efecto_convencion:+.2f} %")
print(f"  El MSE de prueba del OLS es invariante a la escala: "
      f"{mean_squared_error(yte, ols_esl.predict(Xte)):.4f} en ambos casos "
      f"({mean_squared_error(yte, ols_solo_train.predict(Xte_only)):.4f}).")

**📖 Cómo se lee.** El train escalado queda en media ≈ 0 y varianza ≈ 1 con ambos criterios: la estandarización está bien aplicada. **⚠️ Dos cantidades distintas que se confunden a menudo:** (a) el criterio `ddof` —`StandardScaler` divide por la desviación **poblacional** y `scale` de R por la **muestral**— mueve los coeficientes un factor √(96/97) = 0.994832, es decir **0.517 %**, sin cambiar de convención; (b) **cambiar de convención** de estandarización (las 97 filas de ESL frente al escalador ajustado solo con el train) mueve el coeficiente de `lcavol` de 0.6760 a 0.7110, un **+5.18 %**, y es lo que saca a la réplica del rango 0.65–0.71. El MSE de prueba del OLS no cambia en ninguno de los dos casos: es invariante a la escala. Sin estandarizar, la selección del LASSO quedaría dominada por las variables de mayor escala numérica, no por la señal (**⚠️** error operativo número uno). En el laboratorio de Communities el escalador se ajusta **solo con el train** dentro de un `Pipeline` para evitar fuga de información. Desarrollo en la guía de supuestos de la sesión («Sección 2.1»).

**🔎 Qué hace este código.** Diagnostica los requisitos 2.2 y 2.3: traza la **curva de validación** —MSE de CV frente a λ, con banda de ±1 se (se=sd/√k)— del LASSO sobre el train de `prostate`, con los **pliegues barajados** (`cv_barajada`, la misma corrección del Paso 5: el archivo viene ordenado por `lpsa`). La forma en U localiza la sobre-regularización (λ grande) y la infra-regularización (λ pequeño).

In [ ]:
# Diagnostico 2.2/2.3 - curva de validacion: MSE de CV vs lambda (NO escribe en el Excel)
from sklearn.model_selection import validation_curve

rejilla = np.logspace(-3, 0.3, 20)
tr_sc, va_sc = validation_curve(Lasso(max_iter=200000), Xtr, ytr, param_name="alpha",
                                param_range=rejilla, cv=cv_barajada, scoring="neg_mean_squared_error")
mse_tr_vc, mse_va_vc = -tr_sc.mean(1), -va_sc.mean(1)
se_va = (-va_sc).std(1) / np.sqrt(va_sc.shape[1])
i_opt = int(np.argmin(mse_va_vc))
print(f"lambda de menor error de CV = {rejilla[i_opt]:.4f}  (MSE_CV = {mse_va_vc[i_opt]:.3f})")
print(f"MSE_CV con lambda pequeno = {mse_va_vc[0]:.3f}   con lambda grande = {mse_va_vc[-1]:.3f}  (sube -> infra-ajuste)")

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(rejilla, mse_tr_vc, "o-", color=UPC_TINTA, label="MSE de entrenamiento")
ax.plot(rejilla, mse_va_vc, "s-", color=UPC_ROJO, label="MSE de validacion (CV, pliegues barajados)")
ax.fill_between(rejilla, mse_va_vc - se_va, mse_va_vc + se_va, color=UPC_ROJO, alpha=0.15, label="± 1 error estándar (se=sd/√k)")
ax.axvline(rejilla[i_opt], color=UPC_GRIS, ls="--", label=f"lambda optimo = {rejilla[i_opt]:.3f}")
ax.set_xscale("log"); ax.set_xlabel("lambda (escala log)"); ax.set_ylabel("MSE")
ax.set_title("Curva de validacion del LASSO (prostate train)"); ax.legend(fontsize=8)
mostrar(fig, FIGURAS / "S05_sup_curva_validacion.png")

**📖 Cómo se lee.** El error de entrenamiento crece con λ (más penalización = peor ajuste in-sample). El de CV traza la **U**: con λ pequeño el modelo se acerca al OLS y con λ grande sube con claridad (**infra-ajuste**: aquí pasa de ≈ 0.54 a ≈ 1.47, con el mínimo en λ ≈ 0.011 y MSE de CV ≈ 0.536). El mínimo marca el mejor compromiso; sin la banda de ±1 se no se podría elegir λ_1se. **⚠️** Los pliegues van barajados: sin barajar, sobre este archivo ordenado por `lpsa`, la misma curva se desplazaba hacia arriba (mínimo ≈ 0.756). Interpretación en la guía de supuestos de la sesión («Sección 2.2», «Sección 2.3» y «Sección 2.5»).

**🔎 Qué hace este código.** Diagnostica el sesgo-varianza (2.3) con la **curva de aprendizaje**: MSE de entrenamiento y de CV frente al **tamaño de muestra**, con λ fijo (el del LASSO ralo). Si la brecha se mantiene grande, domina la varianza y **más datos ayudan**.

In [ ]:
# Diagnostico 2.3 - curva de aprendizaje: error vs tamano de muestra (NO escribe en el Excel)
from sklearn.model_selection import learning_curve, KFold

tam, tr_ls, va_ls = learning_curve(
    Lasso(alpha=0.2203, max_iter=200000), Xtr, ytr,
    train_sizes=np.linspace(0.3, 1.0, 6), cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="neg_mean_squared_error")
mse_tr_lc, mse_va_lc = -tr_ls.mean(1), -va_ls.mean(1)
print("tamano de train:", tam.tolist())
print("MSE entrenamiento:", np.round(mse_tr_lc, 3).tolist())
print("MSE validacion   :", np.round(mse_va_lc, 3).tolist())

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(tam, mse_tr_lc, "o-", color=UPC_TINTA, label="MSE de entrenamiento")
ax.plot(tam, mse_va_lc, "s-", color=UPC_ROJO, label="MSE de validacion (CV)")
ax.set_xlabel("Tamano del conjunto de entrenamiento"); ax.set_ylabel("MSE")
ax.set_title("Curva de aprendizaje del LASSO (prostate)"); ax.legend()
mostrar(fig, FIGURAS / "S05_sup_curva_aprendizaje.png")

**📖 Cómo se lee.** Con pocos datos la brecha es considerable (MSE de CV ≈ 2.9 con 15 observaciones) y el modelo tiene **alta varianza**; al crecer el train hasta 53 filas el error de CV cae a ≈ 0.68 y la brecha se estrecha. La lección de negocio: cuando domina la varianza, **conseguir más datos** desplaza toda la curva hacia abajo (además de subir λ). Marco en la guía de supuestos de la sesión («Sección 2.3»).

**🔎 Qué hace este código.** Diagnostica el supuesto 2.4: la **estabilidad de la selección** del LASSO. Reejecuta el ajuste con λ vecinos y con remuestreos (bootstrap) del train y observa si el **conjunto seleccionado cambia**. Variables correlacionadas que «se turnan» delatan la inestabilidad que corrige el Elastic Net.

In [ ]:
# Diagnostico 2.4 - estabilidad de la seleccion del LASSO (NO escribe en el Excel)
def activas(alpha, Xt, yt):
    coef = Lasso(alpha=alpha, max_iter=200000).fit(Xt, yt).coef_
    return set(np.array(predictores)[np.abs(coef) > 1e-8])

print("Seleccion con lambda vecinos:")
for a in (0.18, 0.2203, 0.26):
    print(f"  lambda = {a:<6} -> {sorted(activas(a, Xtr, ytr))}")

rng_bs = np.random.default_rng(0)
conjuntos = []
for _ in range(8):
    idx = rng_bs.choice(len(ytr), len(ytr), replace=True)      # remuestreo bootstrap del train
    conjuntos.append(frozenset(activas(0.2203, Xtr[idx], ytr[idx])))
siempre = set.intersection(*[set(c) for c in conjuntos])
print(f"\n{len(set(conjuntos))} conjuntos distintos en 8 remuestreos.")
print(f"Variables que sobreviven SIEMPRE: {sorted(siempre)}")

**📖 Cómo se lee.** Con λ = 0.18 el LASSO deja 5 variables; con λ = 0.22–0.26, solo `{lcavol, lweight, svi}`: la selección **depende de λ**. Bajo bootstrap, `svi`, `lbph` y `pgg45` entran y salen, pero `lcavol` y `lweight` sobreviven **siempre** → el núcleo es robusto, los márgenes no. **⚠️** Reportar «el LASSO eligió estas variables» sin decir a qué λ, ni verificar estabilidad, es un error. La corrección es **Elastic Net** (efecto de agrupamiento). Fuente y método en la guía de supuestos de la sesión («Sección 2.4»).

## Práctica — Drills (ejercicios) (Sección 9 del cuaderno)

Resolver en este mismo notebook, reutilizando los objetos ya cargados. Se entregan los **enunciados**; la solución es parte del trabajo del estudiante. Los enunciados de aquí son **literalmente** los de `evaluacion/drills.docx` (mismos parámetros, mismos rangos): si difieren, manda `evaluacion/drills.docx`.

**Drill 1 — El LASSO lleva coeficientes exactamente a cero.**
Sobre `prostate`, ajustar el LASSO para una malla creciente de λ y graficar el **número de coeficientes distintos de cero** frente a λ. Identificar el orden en que las variables salen del modelo y confirmar que `lcavol` es de las últimas en anularse.

**Drill 2 — Elastic Net frente a LASSO con predictores muy correlacionados.**
Construir dos réplicas ruidosas de `lcavol` (ruido normal de media 0 y desviación **0.05** sobre la columna ya estandarizada, con `np.random.default_rng(semilla)` y dos extracciones consecutivas). Ajustar LASSO puro (`Lasso(alpha=0.05)`) y Elastic Net (`ElasticNet(alpha=0.05, l1_ratio=0.3)`) y comparar cómo reparte cada uno el peso entre las variables gemelas, repitiendo con las semillas `{1, 7, 42, 100, 2024}`. Justificar por qué el Elastic Net es más estable (efecto de agrupamiento).

**Drill 3 — Elegir el grado de un polinomio por CV evitando el sobreajuste.**
Tomar una variable continua (`lcavol` para predecir `lpsa`), ajustar polinomios de **grado 1 a 7** y elegir el grado por validación cruzada 10-fold con pliegues barajados (`KFold(10, shuffle=True, random_state=42)`). Mostrar que el error de entrenamiento cae siempre, pero que el mínimo del error de CV **no basta**: hay que calcular la banda de un error estándar ($\text{se}=\text{sd}/\sqrt{k}$) y decir cuántos grados caen dentro.

---

> **Trazabilidad de las tablas del enunciado.** Las tablas que acompañan a estos drills en `evaluacion/drills.docx` se recomputan **en este mismo cuaderno**, con **exactamente los mismos parámetros** que se acaban de enunciar: la construcción de las gemelas ruidosas del Drill 2 (sd 0.05, `l1_ratio = 0.3`, cinco semillas) está en la celda de registro del Drill 2 → hoja `drill2_gemelas`; y la tabla de grados del Drill 3 (grados 1 a 7, siete semillas), con su error estándar y su dependencia de la semilla, en la celda de registro del Drill 3 → hoja `drill3_grado_semillas`. Ninguna cifra del enunciado carece de camino de recómputo.


## 5.9 — ¿Qué no se puede afirmar, y qué sigue en S06? Cierre (Sección 10 del cuaderno)

### Entregable evaluable
Comparar **OLS / RIDGE / LASSO / Elastic Net** sobre **Communities and Crime** (alta dimensión), eligiendo λ y α por validación cruzada, reportando el **error de prueba** y las **variables seleccionadas por el LASSO**, con una **recomendación** que equilibre interpretabilidad y desempeño. Calificación vigesimal (0–20); rúbrica en `evaluacion/entregable.docx`.

### Control corto
La sesión cierra con un control corto (banco de ítems `[S05]`) sobre: compromiso sesgo-varianza, diferencia L1/L2, lectura de un *path* y de la curva de CV, y cuándo preferir Elastic Net o un GAM.

### Proyecto integrador
Esta sesión alimenta la fase de **Modelado**: en el dataset del proyecto se usará el LASSO para **seleccionar variables** y la CV para elegir la fuerza de la regularización, justificando el modelo final por interpretabilidad y desempeño.

### Materiales de apoyo de la sesión
- Guía del laboratorio: `laboratorio/GUIA_LABORATORIO_S05.docx`
- Plantilla de comparación de modelos: `plantillas/comparacion_modelos_regularizacion.docx`
- Guía visual L1 vs L2: `plantillas/guia_L1_vs_L2.docx`
- Drills y entregable: `evaluacion/drills.docx`, `evaluacion/entregable.docx`
- Mapa de celdas del cuaderno: el cuaderno de la sesión
- Supuestos de la sesión (fuente canónica): la guía de supuestos de la sesión (ver «Sección 8»)
- Definiciones con fórmula y ejemplo numérico: el glosario de la sesión (VIF, Pearson, MSE/R², se = sd/√k, bootstrap)

### Para seguir explorando (actualidad — las **cinco** fuentes integradas de las fuentes de actualidad de la sesión)
1. **LASSO para selección de variables en riesgo de crédito corporativo** — *Enhancing Credit Risk Prediction* (arXiv 2509.22381, 2025): el LASSO como paso de selección estándar en modelos de *default*.
2. **Reducir de 87 a 10 variables para cumplir la regulación de scoring** — *Enhancing ML Interpretability for Credit Scoring* (arXiv 2509.11389, 2025): mismo desempeño con −88.5 % de variables.
3. **Regularización L1 para carteras ralas** — *Sparse behavioral portfolio optimization* (J. Ind. Manag. Optim., 2026): la penalización L1 reduce rotación y costes de transacción.
4. **Elegir λ sin datos de validación (modelos frugales)** — *Validation-Free Sparse Learning* (arXiv 2411.17180, 2024/25): modelos ralos e interpretables como agenda de *green AI*.
5. **Interpretabilidad como requisito regulatorio** — *AI Explainability & Transparency Guide 2026* (GLACIS): modelos simples (regularizados, GAM) como *challenger* y exigencia de cumplimiento.

> Las mismas cinco, en el mismo orden, aparecen en el slide «La regulación premia modelos ralos y auditables» del deck. El archivo de fuentes conserva además una referencia de **contexto no integrada** (IBTimes, 23/11/2024), fuera de la ventana de 18 meses y marcada como tal.

> **Alcance.** La selección de variables de esta sesión se hace por **LASSO** (descarta predictores originales conservando su interpretación). La **reducción de dimensiones** (PCA, análisis factorial, t-SNE, UMAP) pertenece a la Sesión 6 y aquí no se usa. Las sesiones S06–S14 (clustering, asociación, GLM, series, causalidad, supervivencia, recomendación) solo se nombran.

## Transversal — Anexos: la versión extendida de la descomposición (Sección 11 del cuaderno)  <a id="anexos"></a>

Los bloques 🖐️ / 🧮 / ✅ / 🧱 del cuerpo del cuaderno se quedan en la **versión mínima**: la que cabe en clase y la que se evalúa. Aquí se guarda su **versión extendida**, opcional, para quien quiera ver el mecanismo completo. Nada de esta sección escribe en `resultados/S05_resultados.xlsx` ni cambia ninguna cifra del contrato: son verificaciones adicionales sobre los objetos ya construidos.

| Anexo | Extiende a | Qué añade |
|---|---|---|
| **A** | 🖐️ soft-threshold de **una** coordenada (celda 27) | el **descenso por coordenadas completo**: el algoritmo de glmnet sobre los 8 predictores, iterado hasta converger y verificado contra `Lasso` de sklearn |
| **B** | Paso 5, λ_min y λ_1se (celda 72) | la regla de un error estándar **calculada manualmente** desde `mse_path_`, con la banda $\text{se}=\text{sd}/\sqrt{k}$ tabulada λ a λ |
| **C** | 🔬 estabilidad de la selección (celda 148) | el **bootstrap con B = 1000** y su **intervalo percentil** al 95 %, que con los 8 remuestreos del diagnóstico no se podía calcular |

### Anexo A — Descenso por coordenadas completo (extiende la celda 27)

**🔎 Qué hace este código.** Implementa desde cero el algoritmo que resuelve el LASSO —**descenso por coordenadas**, el de `glmnet`— sobre los 8 predictores de `prostate`: en cada pasada recorre las coordenadas una a una, calcula el **residuo parcial** $\mathbf{r}_{-j}$, obtiene la señal $\rho_j=\tfrac{1}{n}\mathbf{x}_j^{\top}\mathbf{r}_{-j}$ y aplica el soft-threshold $S(\rho_j,\lambda)$ de la celda 25. Repite hasta que los coeficientes dejan de moverse y comprueba con `assert` que el resultado coincide con `Lasso` de sklearn. Un detalle que hay que respetar para que coincida: sklearn **centra** $\mathbf{X}$ y $\mathbf{y}$ porque estima el intercepto aparte y **no lo penaliza**; el bucle hace lo mismo. (Sin centrar, la diferencia aparece en el tercer decimal: las columnas de `Xtr` están escaladas sobre las **97** filas —convención ESL— y por eso su media sobre el train de 67 no es exactamente 0.) Es la generalización de la celda 27, que hacía lo mismo con **una** sola coordenada.

In [ ]:
# ANEXO A: descenso por coordenadas del LASSO desde cero (NO escribe en el Excel)
def lasso_coordenadas(Xm, yv, lam, iteraciones=1000, tol=1e-12):
    """LASSO por descenso por coordenadas con la normalizacion 1/(2n) de sklearn.

    Se CENTRAN X e y (igual que `Lasso(fit_intercept=True)`): el intercepto se estima
    aparte y queda fuera del peaje L1. Sin centrar X, las columnas de `Xtr` no tienen
    media exactamente 0 —estan escaladas sobre las 97 filas, convencion ESL— y el
    resultado se aparta de sklearn en el tercer decimal.
    """
    n, p = Xm.shape
    Xc_ = Xm - Xm.mean(axis=0)                 # predictores centrados
    yc_ = yv - yv.mean()                       # respuesta centrada
    beta = np.zeros(p)
    normas = (Xc_ ** 2).sum(axis=0) / n        # ~1 si la columna esta estandarizada
    for _ in range(iteraciones):
        beta_previo = beta.copy()
        for j in range(p):
            r_parcial = yc_ - Xc_ @ beta + Xc_[:, j] * beta[j]   # residuo SIN la variable j
            rho_j = (Xc_[:, j] @ r_parcial) / n                  # la senal de la coordenada j
            beta[j] = np.sign(rho_j) * max(abs(rho_j) - lam, 0.0) / normas[j]
        if np.max(np.abs(beta - beta_previo)) < tol:
            break
    return beta

LAM_ANEXO = 0.05
beta_cd = lasso_coordenadas(Xtr, ytr, LAM_ANEXO)
beta_sk_a = Lasso(alpha=LAM_ANEXO, max_iter=200000, tol=1e-12).fit(Xtr, ytr).coef_

comparativa_a = pd.DataFrame({"descenso_coordenadas": np.round(beta_cd, 4),
                              "sklearn_Lasso": np.round(beta_sk_a, 4)}, index=predictores)
print(comparativa_a.to_string())
print(f"\nMaxima diferencia = {np.max(np.abs(beta_cd - beta_sk_a)):.2e}")
assert np.allclose(beta_cd, beta_sk_a, atol=1e-6), "el descenso por coordenadas debe igualar a sklearn"
print("OK: el LASSO de sklearn ES este bucle de soft-thresholds, coordenada a coordenada.")

**📖 Cómo se lee.** Las dos columnas coinciden hasta la cuarta cifra: `sklearn` no hace nada distinto de este bucle. Se ve además la selección en acción —las coordenadas cuya señal $\lvert\rho_j\rvert$ no superó $\lambda=0.05$ quedan en `0.0` exacto— y por qué el algoritmo necesita el **residuo parcial**: cada coordenada se optimiza *dado* lo que ya explican las demás, y por eso hay que iterar hasta converger en lugar de aplicar el soft-threshold una sola vez.

### Anexo B — La regla de un error estándar, tabulada manualmente (extiende la celda 72)

**🔎 Qué hace este código.** Reconstruye λ_min y λ_1se **sin usar ningún atajo**: toma la matriz `mse_path_` del `LassoCV` ya ajustado (un error por λ y por pliegue), promedia entre pliegues, calcula la banda $\text{se}=\text{sd}/\sqrt{k}$ —**no** la sd cruda—, fija el techo `CV(λ_min) + se(λ_min)` y busca el **mayor** λ que cabe debajo. Imprime la tabla alrededor de la zona de decisión, que es lo que hay que saber leer en el control corto y en el laboratorio de Communities.

In [ ]:
# ANEXO B: lambda_min y lambda_1se a mano desde mse_path_ (NO escribe en el Excel)
k_pliegues = lasso_cv.mse_path_.shape[1]
cv_media = lasso_cv.mse_path_.mean(axis=1)
cv_sd = lasso_cv.mse_path_.std(axis=1)
cv_se = cv_sd / np.sqrt(k_pliegues)                      # se = sd / raiz(k)  <- NO la sd cruda
j_min = int(np.argmin(cv_media))
techo = cv_media[j_min] + cv_se[j_min]
lam_1se_mano = lasso_cv.alphas_[cv_media <= techo].max()

tabla_b = pd.DataFrame({"lambda": lasso_cv.alphas_, "CV_MSE": cv_media,
                        "sd_entre_pliegues": cv_sd, "se": cv_se,
                        "cabe_bajo_el_techo": cv_media <= techo})
foco = tabla_b.iloc[max(j_min - 3, 0): j_min + 4]        # ventana alrededor del minimo
print(f"k = {k_pliegues} pliegues barajados")
print(f"lambda_min = {lasso_cv.alphas_[j_min]:.4f}   CV-MSE = {cv_media[j_min]:.4f}   "
      f"sd = {cv_sd[j_min]:.4f}   se = sd/raiz(k) = {cv_se[j_min]:.4f}")
print(f"techo de la banda = {cv_media[j_min]:.4f} + {cv_se[j_min]:.4f} = {techo:.4f}")
print(f"lambda_1se (el MAYOR lambda bajo el techo) = {lam_1se_mano:.4f}\n")
print(foco.round(4).to_string(index=False))
techo_sd_cruda = cv_media[j_min] + cv_sd[j_min]
print(f"\nSi se usara la sd CRUDA ({cv_sd[j_min]:.4f}) el techo seria {techo_sd_cruda:.4f} "
      f"y lambda_1se saldria {lasso_cv.alphas_[cv_media <= techo_sd_cruda].max():.4f}: "
      "una banda mas ancha y un modelo mas ralo que la evidencia no respalda.")
assert abs(lam_1se_mano - lambda_1se) < 1e-12, "el lambda_1se a mano debe igualar al de la celda 72"
print("OK: reproduce el lambda_1se del Paso 5.")

**📖 Cómo se lee.** La columna `cabe_bajo_el_techo` es la regla entera: se recorre la malla de λ de mayor a menor y se toma el **primero** que entra. La comparación final es la lección que el Drill 3 y el Nivel 4 de la guía evalúan: usar la **sd entre pliegues** en lugar del **error estándar del promedio** infla la banda en un factor $\sqrt{k}$ (con $k=10$, más del triple) y «justifica» un modelo más ralo de lo que los datos sostienen. El `assert` confirma que este cálculo manual devuelve el mismo λ_1se que la celda 72.

### Anexo C — Bootstrap con B = 1000 e intervalo percentil (extiende la celda 148)

**🔎 Qué hace este código.** El diagnóstico de la «Sección 8» usa **8** remuestreos: bastan para *exhibir* que la selección del LASSO se mueve, pero son demasiado pocos para un intervalo. Aquí se ejecuta el **bootstrap completo**: `B = 1000` remuestreos con reemplazo del train de 67 filas, reajustando el LASSO con el λ operativo de la réplica (0.2203). De cada remuestreo se guarda el coeficiente de `lcavol` y qué variables quedan activas, y con esa distribución se construye el **intervalo percentil al 95 %** —los cuantiles 2,5 y 97,5— y la **frecuencia de selección** de cada predictor.

⚠️ Un coeficiente regularizado está sesgado a propósito: este intervalo describe la **variabilidad muestral del procedimiento**, no es un intervalo de confianza clásico ni admite lectura inferencial.

In [ ]:
# ANEXO C: bootstrap B=1000 e intervalo percentil (NO escribe en el Excel)
B_BOOT = 1000
LAM_BOOT = 0.2203                                   # lambda operativo de la replica (celda 78)
rng_boot = np.random.default_rng(RANDOM_STATE)
n_tr = len(ytr)
coef_lcavol_boot = np.empty(B_BOOT)
veces_activa = np.zeros(len(predictores))

for b in range(B_BOOT):
    idx = rng_boot.choice(n_tr, n_tr, replace=True)  # remuestreo CON reemplazo
    coef_b = Lasso(alpha=LAM_BOOT, max_iter=200000).fit(Xtr[idx], ytr[idx]).coef_
    coef_lcavol_boot[b] = coef_b[predictores.index("lcavol")]
    veces_activa += (np.abs(coef_b) > 1e-8)

ic_bajo, ic_alto = np.percentile(coef_lcavol_boot, [2.5, 97.5])
frecuencia = pd.Series(veces_activa / B_BOOT, index=predictores).sort_values(ascending=False)

print(f"B = {B_BOOT} remuestreos, lambda = {LAM_BOOT}, train de {n_tr} filas")
print(f"coef de lcavol: media {coef_lcavol_boot.mean():.4f} — se_bootstrap {coef_lcavol_boot.std(ddof=1):.4f}")
print(f"IC PERCENTIL 95 % = [{ic_bajo:.4f}, {ic_alto:.4f}]   (cuantiles 2,5 y 97,5)")
print("\nFrecuencia de seleccion por predictor (proporcion de remuestreos en que sobrevive):")
print(frecuencia.round(3).to_string())
assert ic_bajo < coef_lcavol_boot.mean() < ic_alto, "el IC percentil debe contener a la media bootstrap"
print("\nOK: con B=1000 el intervalo percentil ya esta definido; con los 8 remuestreos de la celda 148, no.")

**📖 Cómo se lee.** El intervalo percentil da la anchura real de la incertidumbre del coeficiente de `lcavol` bajo remuestreo, y la tabla de frecuencias responde a la pregunta de negocio que sigue a toda selección: *«si volviera a tomar la muestra, ¿saldría la misma lista?»*. Los predictores con frecuencia cercana a 1 son los que se pueden defender ante un comité; los de frecuencia intermedia son los que el LASSO elige «a suertes» entre gemelos y para los que la sesión recomienda **Elastic Net** o el criterio de dominio. Contraste con la celda 148: con $B=8$ solo se podía decir «la selección cambia»; con $B=1000$ se puede **cuantificar** cuánto.

> **Cierre de los anexos.** Nada de esta sección altera el contrato de la sesión: el Excel se escribe una sola vez (celda 111) y el validador el material de referencia de la sesión recomputa desde los CSV crudos. Estos tres anexos son material **opcional** de profundización; su lugar en la clase es la consulta posterior, no los 90 minutos.